# OLiMPiC Pitch Correction — Experiment Notebook

Unified entry point for training, evaluation, and baseline comparison.
Run cells in order from a fresh kernel.

**Cell tags:** `⚙️` config · `⚡` run-once setup · `📊` display (no writes) · `⚠️` writes or trains

### Historical runs

| Run | Train | Criterion | Best dev top-1 |
|-----|-------|-----------|----------------|
| correction_v1 | 3K | loss | 30/499 = 6.01% |
| correction_v2 | 3K | loss | 44/499 = 8.82% |
| correction_v3 | 10K | loss | 55/499 = 11.02% |
| correction_v3_top1 | 10K | top-1 | 47/499 = 9.42% |
| correction_v4_mask | 10K | loss | 46/499 = 9.22% |
| correction_v4_mask_top1 | 10K | top-1 | 63/499 = 12.63% |

Baselines: MFC_10k 26/499 = 5.21% · voice_mfc 46/499 = 9.22% · prev_pitch 20/499 · next_pitch 17/499

### v5 / v6 experiments (seed 42, MPS)

| Run | Arch | Params | Best ep | Dev top-1 | Dev top-5 | Wall (s) |
|-----|------|--------|---------|-----------|-----------|----------|
| v5_v4_control | 128d·3L·512ff | 657,972 | 4/9 | 57/499=11.42% | 34.87% | 46 |
| **v5_voice_conditioned** | 128d·3L·512ff + voice | 659,924 | 10/15 | **83/499=16.63%** | **52.30%** | 74 |
| v6_voice_large | 256d·4L·1024ff + voice | 3,287,252 | 14/19 | 56/499=11.22% | 41.08% | 211 |
| v7_dynamic_mask | 128d·3L·512ff + voice | 659,924 | 1/6 | 55/499=11.02% | 39.28% | 38 |

**v5 is the current best.** v7 (dynamic masking) = 11.02% — worse than v5 static; early stop at ep 1. v6 underfits (train top-1 = 8% < dev 11%) — 3.3M params under-served by 10K examples at lr=1e-3.


In [ ]:
# ⚙️  CONFIG — edit this cell to switch experiments
# ─────────────────────────────────────────────────────────────────────────────
from pathlib import Path
import json

# Auto-detect ml/ directory (works whether Jupyter is launched from project root or ml/)
_cwd = Path.cwd()
for _c in [_cwd / "ml", _cwd, _cwd.parent / "ml"]:
    if (_c / "data" / "processed").exists():
        ML_ROOT = _c.resolve()
        break
else:
    raise RuntimeError(
        f"Cannot locate ml/ directory. Launch Jupyter from the project root.\n"
        f"cwd={_cwd}"
    )
print(f"ML_ROOT: {ML_ROOT}")

DATASET_ROOT = ML_ROOT / "dataset/olimpic-1.0-synthetic"
DATA_DIR     = ML_ROOT / "data"
RUNS_DIR     = DATA_DIR / "runs"

DEV_JSONL  = DATA_DIR / "processed/correction_dev_v1.jsonl"
TRAIN_10K  = DATA_DIR / "processed/correction_train_10k.jsonl"
TRAIN_3K   = DATA_DIR / "processed/correction_train_v1.jsonl"
CONTEXT_BASELINE_JSON = RUNS_DIR / "context_baselines/context_baseline_results.json"

# All existing run directories
RUNS = {
    "v1":              RUNS_DIR / "correction_v1",
    "v2":              RUNS_DIR / "correction_v2",
    "v3":              RUNS_DIR / "correction_v3",
    "v3_top1":         RUNS_DIR / "correction_v3_top1",
    "v4_mask":         RUNS_DIR / "correction_v4_mask",
    "v4_mask_top1":    RUNS_DIR / "correction_v4_mask_top1",
    "v5_v4_control":   RUNS_DIR / "v5_v4_control",
    "v5_voice":        RUNS_DIR / "v5_voice_conditioned",
    "v6_voice_large":  RUNS_DIR / "v6_voice_large",
    "v7_dynamic_mask": RUNS_DIR / "v7_dynamic_mask",
}

# ── New experiment config ─────────────────────────────────────────────────────
# Edit this dict, then run the training cell (Section 3).
# To switch between sizes:
#   small (v5):  d_model=128, num_layers=3, dim_ff=512
#   large (v6):  d_model=256, num_layers=4, dim_ff=1024
EXP = {
    "name":               "correction_v7",  # → RUNS_DIR / name  (must not already exist)
    "train_jsonl":        TRAIN_10K,
    "use_mask":           True,
    "voice_conditioning": True,
    "voice_dim":          32,
    "d_model":            256,    # 128 = small  |  256 = large
    "num_layers":         4,      # 3   = small  |  4   = large
    "dim_ff":             1024,   # 512 = small  |  1024 = large
    "checkpoint_by":      "top1",
    "seed":               42,
    "batch_size":         32,
    "lr":                 1e-3,
    "weight_decay":       0.01,
    "max_epochs":         30,
    "patience":           5,
    "time_limit":         1800,
    "device":             None,   # None = auto  |  "cpu" = deterministic
}
EXP["output_dir"] = RUNS_DIR / EXP["name"]
print(f"New experiment → {EXP['output_dir']}")


In [ ]:
import re, csv, random, sys, time
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

MAX_LEN            = 257
PAD_TOKEN          = "<PAD>"
UNK_TOKEN          = "<UNK>"
MASK_TOKEN         = "[MASK]"
PITCH_RE           = re.compile(r'^[A-G][0-9]$')
N_HEADS            = 4          # fixed across all experiments
DROPOUT_EXPERIMENT = 0.1


## Core Library

Define all functions once; reused by setup, results, and training cells.

In [ ]:
# Vocabulary builders

def build_vocab_from_train(dataset_root, train_ids):
    """Scan train LMX files; returns {token: id} with PAD=0, UNK=1."""
    raw = set()
    for sid in train_ids:
        p = dataset_root / f"{sid}.lmx"
        if p.exists():
            raw.update(p.read_text().split())
    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for t in sorted(raw):
        vocab[t] = len(vocab)
    return vocab


def build_pitch_classes(train_ids, dataset_root):
    """Return {pitch_token: class_id} from train split (sorted for determinism)."""
    pitches = set()
    for sid in train_ids:
        p = dataset_root / f"{sid}.lmx"
        if p.exists():
            pitches.update(t for t in p.read_text().split() if PITCH_RE.match(t))
    return {p: i for i, p in enumerate(sorted(pitches))}


def add_mask_to_vocab(vocab):
    """Append [MASK] as the last vocab entry (ID = len(vocab))."""
    assert MASK_TOKEN not in vocab
    v = dict(vocab)
    v[MASK_TOKEN] = len(vocab)
    return v


In [ ]:
# Windowing and LMX voice-map extraction

def window_tokens(tokens, error_pos, replacement_token, max_len=MAX_LEN):
    """
    Crop to at most max_len tokens centred on error_pos.
    Returns (window, window_error_pos, window_lo).
    Invariant: window[window_error_pos] == replacement_token.
    """
    n = len(tokens)
    if n <= max_len:
        lo, window = 0, tokens[:]
    else:
        lo     = max(0, min(error_pos - max_len // 2, n - max_len))
        window = tokens[lo : lo + max_len]
    w_pos = error_pos - lo
    if window[w_pos] != replacement_token:
        raise ValueError(
            f"Window invariant broken: error_pos={error_pos}, lo={lo}, "
            f"got={window[w_pos]!r}, expected={replacement_token!r}"
        )
    return window, w_pos, lo


def build_voice_map(tokens):
    """
    For each pitch position i in tokens, return its LMX voice label (str or None).

    LMX convention: standalone notes have voice:N immediately after the pitch token.
    Chord notes (preceded by 'chord') inherit the current running voice.
    Orphaned voice:N tokens at the window start update current_voice.
    """
    voice_map, current_voice = {}, None
    for i, tok in enumerate(tokens):
        if PITCH_RE.match(tok):
            nxt = tokens[i + 1] if i + 1 < len(tokens) else ""
            if nxt.startswith("voice:"):
                current_voice = nxt
            voice_map[i] = current_voice
        elif tok.startswith("voice:") and (i == 0 or not PITCH_RE.match(tokens[i - 1])):
            current_voice = tok
    return voice_map


In [ ]:
# CorrectionExample and JSONL loading

class CorrectionExample:
    """One processed example, ready for batching."""
    __slots__ = ("sample_id", "token_ids", "padding_mask", "window_error_pos",
                 "target_class", "window_lo", "target_token", "replacement_token",
                 "voice_label", "voice_id")

    def __init__(self, sample_id, token_ids, padding_mask, window_error_pos,
                 target_class, window_lo, target_token, replacement_token,
                 voice_label=None, voice_id=0):
        self.sample_id, self.token_ids, self.padding_mask = sample_id, token_ids, padding_mask
        self.window_error_pos, self.target_class, self.window_lo = window_error_pos, target_class, window_lo
        self.target_token, self.replacement_token = target_token, replacement_token
        self.voice_label, self.voice_id = voice_label, voice_id


def build_voice_vocab(examples):
    """
    Build {voice_label_str: id} from train examples.
    ID 0 is reserved for UNKNOWN/None; labels start at 1.
    Pass only train examples; dev labels absent from vocab map to 0.
    """
    labels = sorted(set(e.voice_label for e in examples if e.voice_label is not None))
    return {label: i + 1 for i, label in enumerate(labels)}


def load_all_examples(jsonl_path, vocab, pitch_classes, expected_id_set,
                      split_label, mask=False, voice_vocab=None):
    """
    Load every line of a JSONL as CorrectionExample objects.

    mask=True     : replace replacement_token with MASK_TOKEN before tokenisation.
    voice_vocab   : if provided, map voice_label → voice_id (0 = UNKNOWN/None).
                    Voice is parsed BEFORE masking so the pitch token is present.
    Returns (examples, unk_count, total_window_token_count).
    """
    unk_id, pad_id = vocab[UNK_TOKEN], vocab[PAD_TOKEN]
    examples, unk_count, total_count = [], 0, 0
    with open(jsonl_path) as f:
        for lineno, line in enumerate(f, 1):
            ex  = json.loads(line)
            sid = ex["sample_id"]
            if sid not in expected_id_set:
                raise ValueError(f"line {lineno}: {sid} not in {split_label} split")
            tgt = ex["target_token"]
            if tgt not in pitch_classes:
                raise ValueError(f"line {lineno}: target {tgt!r} not in pitch_classes")
            rep    = ex["replacement_token"]
            window, w_pos, w_lo = window_tokens(ex["input_tokens"], ex["error_position"], rep)
            # Parse voice BEFORE masking (replacement token is still a pitch token here)
            voice_label = build_voice_map(window).get(w_pos)
            voice_id    = 0 if voice_vocab is None else voice_vocab.get(voice_label, 0)
            if mask:
                window = list(window)
                assert window[w_pos] == rep
                window[w_pos] = MASK_TOKEN
            ids     = [vocab.get(t, unk_id) for t in window]
            pad_len = MAX_LEN - len(ids)
            unk_count  += ids.count(unk_id)
            total_count += len(ids)
            examples.append(CorrectionExample(
                sample_id         = sid,
                token_ids         = ids + [pad_id] * pad_len,
                padding_mask      = [False] * len(ids) + [True] * pad_len,
                window_error_pos  = w_pos,
                target_class      = pitch_classes[tgt],
                window_lo         = w_lo,
                target_token      = tgt,
                replacement_token = rep,
                voice_label       = voice_label,
                voice_id          = voice_id,
            ))
    return examples, unk_count, total_count


def print_dataset_stats(label, examples, unk_tokens, total_tokens, pitch_classes):
    score_ids     = {e.sample_id.split("/")[1] for e in examples}
    target_counts = Counter(e.target_token for e in examples)
    wls           = [sum(1 for m in e.padding_mask if not m) for e in examples]
    unk_rate      = unk_tokens / total_tokens if total_tokens else 0.0
    print(f"\n{'─'*60}\n {label}\n{'─'*60}")
    print(f"  examples : {len(examples)}")
    print(f"  scores   : {len(score_ids)}")
    print(f"  UNK rate : {unk_rate:.4%}  ({unk_tokens}/{total_tokens})")
    print(f"  window   : min={min(wls)}  max={max(wls)}  mean={sum(wls)/len(wls):.1f}")
    print(f"  target top-5: " + "  ".join(f"{p}={n}" for p, n in target_counts.most_common(5)))
    print(f"  classes covered: {len(target_counts)} / {len(pitch_classes)}")


In [ ]:
# CorrectionModel — bidirectional Transformer encoder for pitch correction

class CorrectionModel(nn.Module):
    """
    Minimal bidirectional Transformer encoder.
    Input : corrupted token sequence + known error position.
    Output: logits over pitch classes (extraction at error_pos hidden state).

    voice_conditioning=True: a learned voice embedding is concatenated with
    the error-position hidden state before the classification head.
    """
    def __init__(self, vocab_size, num_pitch_classes, d_model=128, nhead=4,
                 num_layers=3, dim_feedforward=512, dropout=0.1,
                 max_len=MAX_LEN, pad_id=0,
                 voice_conditioning=False, n_voices=0, voice_dim=32):
        super().__init__()
        self.voice_conditioning = voice_conditioning
        self.token_emb  = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_emb    = nn.Embedding(max_len, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=False)
        # enable_nested_tensor=False: avoids MPS-unsupported op (no correctness impact)
        self.encoder    = nn.TransformerEncoder(enc_layer, num_layers=num_layers,
                                                enable_nested_tensor=False)
        if voice_conditioning:
            self.voice_emb  = nn.Embedding(n_voices + 1, voice_dim)  # 0 = UNKNOWN
            classifier_in   = d_model + voice_dim
        else:
            classifier_in   = d_model
        self.classifier = nn.Linear(classifier_in, num_pitch_classes)

    def forward(self, token_ids, error_pos, padding_mask, voice_ids=None):
        """token_ids: (B,L)  error_pos: (B,)  padding_mask: (B,L) True=pad"""
        B, L   = token_ids.shape
        pos    = torch.arange(L, device=token_ids.device).unsqueeze(0).expand(B, -1)
        x      = self.token_emb(token_ids) + self.pos_emb(pos)
        x      = self.encoder(x, src_key_padding_mask=padding_mask)
        hidden = x[torch.arange(B, device=x.device), error_pos]
        if self.voice_conditioning:
            v_emb  = self.voice_emb(voice_ids)        # (B, voice_dim)
            hidden = torch.cat([hidden, v_emb], dim=-1)
        return self.classifier(hidden)


In [ ]:
# Training and evaluation helpers

def collate(examples, device):
    """Collate a list of CorrectionExample into a batch dict of tensors."""
    return {
        "token_ids":      torch.tensor([e.token_ids        for e in examples], dtype=torch.long, device=device),
        "padding_mask":   torch.tensor([e.padding_mask     for e in examples], dtype=torch.bool, device=device),
        "error_pos":      torch.tensor([e.window_error_pos for e in examples], dtype=torch.long, device=device),
        "target_classes": torch.tensor([e.target_class     for e in examples], dtype=torch.long, device=device),
        "voice_ids":      torch.tensor([e.voice_id         for e in examples], dtype=torch.long, device=device),
    }


def run_train_epoch(model, examples, batch_size, optimizer, loss_fn, device, rng):
    """One full training pass. Returns (avg_loss, top1, top5) — example-level aggregation."""
    model.train()
    shuffled = list(examples); rng.shuffle(shuffled)
    loss_sum = top1 = top5 = n = 0
    for i in range(0, len(shuffled), batch_size):
        b    = collate(shuffled[i : i + batch_size], device)
        bs   = b["token_ids"].size(0)
        optimizer.zero_grad()
        lg   = model(b["token_ids"], b["error_pos"], b["padding_mask"], b["voice_ids"])
        loss = loss_fn(lg, b["target_classes"])
        loss.backward(); optimizer.step()
        tgt      = b["target_classes"]
        loss_sum += loss.item() * bs
        top1     += lg.topk(1, dim=-1)[1].squeeze(1).eq(tgt).sum().item()
        top5     += lg.topk(5, dim=-1)[1].eq(tgt.unsqueeze(1)).any(1).sum().item()
        n        += bs
    return loss_sum / n, top1 / n, top5 / n


@torch.no_grad()
def evaluate_dataset(model, examples, batch_size, loss_fn, device):
    """Eval loop. Returns (avg_loss, top1, top5, pred_class_list) — example-level."""
    model.eval()
    loss_sum = top1 = top5 = n = 0; all_preds = []
    for i in range(0, len(examples), batch_size):
        b    = collate(examples[i : i + batch_size], device)
        bs   = b["token_ids"].size(0)
        lg   = model(b["token_ids"], b["error_pos"], b["padding_mask"], b["voice_ids"])
        loss = loss_fn(lg, b["target_classes"])
        tgt  = b["target_classes"]
        loss_sum += loss.item() * bs
        top1     += lg.topk(1, dim=-1)[1].squeeze(1).eq(tgt).sum().item()
        top5     += lg.topk(5, dim=-1)[1].eq(tgt.unsqueeze(1)).any(1).sum().item()
        n        += bs
        all_preds.extend(lg.argmax(dim=-1).tolist())
    return loss_sum / n, top1 / n, top5 / n, all_preds


@torch.no_grad()
def get_logits_for_dataset(model, examples, batch_size, device):
    """Return full logit tensor (N, num_pitch_classes) on CPU."""
    model.eval()
    parts = []
    for i in range(0, len(examples), batch_size):
        b = collate(examples[i : i + batch_size], device)
        parts.append(model(b["token_ids"], b["error_pos"], b["padding_mask"], b["voice_ids"]).cpu())
    return torch.cat(parts, dim=0)


def build_predictions(examples, pred_classes, class_to_pitch, logits_all):
    """Build list of per-example prediction dicts for saving to JSON."""
    return [{
        "sample_id":         e.sample_id,
        "error_position":    e.window_error_pos + e.window_lo,
        "window_error_pos":  e.window_error_pos,
        "window_lo":         e.window_lo,
        "replacement_token": e.replacement_token,
        "target_token":      e.target_token,
        "predicted_token":   class_to_pitch[pc],
        "correct":           pc == e.target_class,
        "top5_candidates":   [class_to_pitch[i] for i in lg.topk(5).indices.tolist()],
        "voice_label":       e.voice_label,
    } for e, pc, lg in zip(examples, pred_classes, logits_all)]


In [ ]:
# Baseline functions (MFC, context rules, per-voice MFC)

def compute_mfc_baseline(train_examples, dev_examples):
    """Most-frequent-class from train frequency, evaluated on dev."""
    counts = Counter(e.target_token for e in train_examples)
    top1   = counts.most_common(1)[0][0]
    top5   = {p for p, _ in counts.most_common(5)}
    h1 = sum(1 for e in dev_examples if e.target_token == top1)
    h5 = sum(1 for e in dev_examples if e.target_token in top5)
    n  = len(dev_examples)
    print(f"  MFC top-1: {h1}/{n} = {h1/n:.2%}  ({top1!r})")
    print(f"  MFC top-5: {h5}/{n} = {h5/n:.2%}  {sorted(top5)}")
    return h1 / n, h5 / n, top1


def context_baseline_eval(dev_jsonl, mfc_token):
    """
    prev_pitch and next_pitch context rules directly from raw JSONL.
    Falls back to mfc_token when no same-voice neighbour found within the window.
    Returns list of per-example result dicts.
    """
    results = []
    with open(dev_jsonl) as f:
        for line in f:
            ex   = json.loads(line)
            inp, pos = ex["input_tokens"], ex["error_position"]
            rep, tgt = ex["replacement_token"], ex["target_token"]
            window, w_pos, _ = window_tokens(inp, pos, rep)
            vmap = build_voice_map(window)
            v    = vmap.get(w_pos)
            prev = next_tok = None
            for i in range(w_pos - 1, -1, -1):
                if i in vmap and vmap[i] == v: prev = window[i]; break
            for i in range(w_pos + 1, len(window)):
                if i in vmap and vmap[i] == v: next_tok = window[i]; break
            results.append({
                "sample_id":  ex["sample_id"], "target": tgt,
                "prev":       prev or mfc_token,  "next": next_tok or mfc_token,
                "prev_found": prev is not None,   "next_found": next_tok is not None,
            })
    return results


def voice_mfc_eval(train_jsonl, dev_jsonl):
    """
    Per-voice most-frequent-class from train, evaluated on dev.
    Falls back to global MFC for unseen voices.
    Returns (results_list, voice_mfc_map, global_mfc_token).
    """
    global_counts = Counter()
    voice_counts  = defaultdict(Counter)
    with open(train_jsonl) as f:
        for line in f:
            ex = json.loads(line)
            w, wp, _ = window_tokens(ex["input_tokens"], ex["error_position"],
                                     ex["replacement_token"])
            v = build_voice_map(w).get(wp)
            global_counts[ex["target_token"]] += 1
            voice_counts[v][ex["target_token"]] += 1
    global_mfc    = global_counts.most_common(1)[0][0]
    voice_mfc_map = {v: c.most_common(1)[0][0] for v, c in voice_counts.items()}
    hits = fallbacks = 0
    results = []
    with open(dev_jsonl) as f:
        for line in f:
            ex = json.loads(line)
            w, wp, _ = window_tokens(ex["input_tokens"], ex["error_position"],
                                     ex["replacement_token"])
            v    = build_voice_map(w).get(wp)
            pred = voice_mfc_map.get(v, global_mfc)
            if v not in voice_mfc_map: fallbacks += 1
            correct = pred == ex["target_token"]
            if correct: hits += 1
            results.append({"sample_id": ex["sample_id"], "correct": correct, "voice": v})
    n = len(results)
    print(f"  voice_mfc: {hits}/{n} = {hits/n:.2%}  (fallback on {fallbacks})")
    return results, voice_mfc_map, global_mfc


## 1. Setup

Run this section first (fresh kernel).
Vocab building scans all train-split LMX files (~10–20 s on first run).


In [ ]:
# ⚡ Load official split IDs and build vocabulary
def _read_split(name):
    return [l.strip() for l in
            (DATASET_ROOT / f"samples.{name}.txt").read_text().splitlines()
            if l.strip()]

train_ids    = _read_split("train")
dev_ids      = _read_split("dev")
train_id_set = set(train_ids)
dev_id_set   = set(dev_ids)
print(f"Split IDs: train={len(train_ids)}  dev={len(dev_ids)}")

print("Building vocab from train split LMX files...")
vocab          = build_vocab_from_train(DATASET_ROOT, train_ids)
pitch_classes  = build_pitch_classes(train_ids, DATASET_ROOT)
class_to_pitch = {v: k for k, v in pitch_classes.items()}
print(f"Vocab: {len(vocab)}  Pitch classes: {len(pitch_classes)}")

DEVICE = (torch.device(EXP["device"]) if EXP["device"] else
          torch.device("cuda" if torch.cuda.is_available() else
                       "mps"  if torch.backends.mps.is_available() else "cpu"))
print(f"Device: {DEVICE}")


In [ ]:
# ⚡ Load train and dev examples (unmasked; training cells handle masking internally)
print("Loading 10K train examples...")
train_examples, train_unk, train_total = load_all_examples(
    TRAIN_10K, vocab, pitch_classes, train_id_set, "train", mask=False)

print("Loading dev examples...")
dev_examples, dev_unk, dev_total = load_all_examples(
    DEV_JSONL, vocab, pitch_classes, dev_id_set, "dev", mask=False)

# Score-level cross-check (no score should appear in both splits)
ts = {e.sample_id.split("/")[1] for e in train_examples}
ds = {e.sample_id.split("/")[1] for e in dev_examples}
assert not (ts & ds), f"Score overlap found: {ts & ds}"
print(f"Score overlap: CLEAN  (train={len(ts)} scores, dev={len(ds)} scores)")


In [ ]:
# 📊 Dataset statistics and MFC baseline
print_dataset_stats("TRAIN (10K)", train_examples, train_unk, train_total, pitch_classes)
print_dataset_stats("DEV   (499)", dev_examples,   dev_unk,   dev_total,   pitch_classes)

print("\nMost-frequent-class baseline (train → dev):")
mfc_top1_rate, mfc_top5_rate, mfc_token = compute_mfc_baseline(train_examples, dev_examples)


## 2. Existing Run Results 📊

Load and compare all saved run artifacts. No model inference — reads JSON files only.
Requires Section 1 (setup) to have run.


In [ ]:
# 📊 Comparison table: actual checkpoint performance (from dev_predictions.json)
# Checkpoint epoch is read directly from best_checkpoint.pt (ground truth).
# Dev top-1 is counted from saved dev_predictions.json (not metrics peak).
all_metrics = {}
for run_name, run_dir in RUNS.items():
    mf = run_dir / "metrics.json"
    if mf.exists():
        all_metrics[run_name] = json.loads(mf.read_text())

import torch as _torch

header = f"{'Run':<22} {'Eps':>4} {'Ckpt':>5} {'Dev top-1':>17} {'Dev top-5':>10}"
sep    = "─" * len(header)
print(header); print(sep)
for rn, metrics in all_metrics.items():
    run_dir = RUNS[rn]
    n_ep    = max(m["epoch"] for m in metrics)

    # Actual checkpoint epoch from the saved checkpoint file
    ckpt_f  = run_dir / "best_checkpoint.pt"
    if ckpt_f.exists():
        ck      = _torch.load(ckpt_f, map_location="cpu", weights_only=False)
        ckpt_ep = ck["epoch"]
    else:
        ckpt_ep = "?"

    # Actual top-1: count from saved predictions (reflects checkpoint, not peak epoch)
    pred_f = run_dir / "dev_predictions.json"
    if pred_f.exists():
        preds   = json.loads(pred_f.read_text())
        n_corr  = sum(1 for p in preds if p["correct"])
        n_tot   = len(preds)
        t1_disp = f"{n_corr}/{n_tot} = {n_corr/n_tot:.2%}"
    else:
        best_row = max(metrics, key=lambda m: m["dev_top1"])
        t1_disp  = f"{best_row['dev_top1']:.2%}  (metrics peak)"

    # Top-5 from the checkpoint epoch row in metrics
    ep_row = next((m for m in metrics if m["epoch"] == ckpt_ep), metrics[-1])
    print(f"{rn:<22} {n_ep:>4} {str(ckpt_ep):>5} {t1_disp:>17} {ep_row['dev_top5']:>10.2%}")

# Baselines from saved context_baseline_results.json
if CONTEXT_BASELINE_JSON.exists():
    cb    = json.loads(CONTEXT_BASELINE_JSON.read_text())
    N     = len(cb)
    mfc_h  = sum(1 for r in cb if r["mfc_correct"])
    prev_h = sum(1 for r in cb if r["prev_correct"])
    next_h = sum(1 for r in cb if r["next_correct"])
    vm_h   = 46   # from voice_mfc_baseline run
    print(sep)
    print(f"{'MFC_10k':<22} {'—':>4} {'—':>5} {mfc_h}/{N} = {mfc_h/N:.2%}")
    print(f"{'voice_mfc':<22} {'—':>4} {'—':>5} {vm_h}/{N} = {vm_h/N:.2%}")
    print(f"{'prev_pitch':<22} {'—':>4} {'—':>5} {prev_h}/{N} = {prev_h/N:.2%}")
    print(f"{'next_pitch':<22} {'—':>4} {'—':>5} {next_h}/{N} = {next_h/N:.2%}")


In [ ]:
# 📊 Training curves for all saved runs (solid=dev, dashed=train)
if not all_metrics:
    print("No metrics loaded. Run Cell 14 first.")
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    colors = plt.cm.tab10.colors

    for (rn, metrics), color in zip(all_metrics.items(), colors):
        epochs = [m["epoch"] for m in metrics]
        ax1.plot(epochs, [m["dev_top1"]   for m in metrics], color=color, lw=2,   label=rn)
        ax1.plot(epochs, [m["train_top1"] for m in metrics], color=color, lw=1, ls="--", alpha=0.5)
        ax2.plot(epochs, [m["dev_loss"]   for m in metrics], color=color, lw=2,   label=rn)
        ax2.plot(epochs, [m["train_loss"] for m in metrics], color=color, lw=1, ls="--", alpha=0.5)

    ax1.set(title="Dev top-1 accuracy (solid=dev  dashed=train)",
            xlabel="Epoch", ylabel="Top-1 accuracy")
    ax2.set(title="Loss (solid=dev  dashed=train)",
            xlabel="Epoch", ylabel="Cross-entropy")
    for ax in (ax1, ax2):
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()


In [ ]:
# 📊 Prediction examples from a saved run
# Change PREVIEW_RUN to any key in RUNS
PREVIEW_RUN = "v4_mask_top1"

preds_file = RUNS[PREVIEW_RUN] / "dev_predictions.json"
if not preds_file.exists():
    print(f"No predictions file for {PREVIEW_RUN}")
else:
    preds     = json.loads(preds_file.read_text())
    n_correct = sum(1 for p in preds if p["correct"])
    n_total   = len(preds)
    print(f"{PREVIEW_RUN}  →  dev top-1: {n_correct}/{n_total} = {n_correct/n_total:.2%}\n")

    print("── Correct predictions (first 5) ──")
    for p in [x for x in preds if x["correct"]][:5]:
        print(f"  ✓ {p['sample_id']}  pos={p['error_position']}")
        print(f"    rep={p['replacement_token']!r}  "
              f"target={p['target_token']!r}  predicted={p['predicted_token']!r}")
        if "top5_candidates" in p:
            print(f"    top-5: {p['top5_candidates']}")

    print("\n── Wrong predictions (first 5) ──")
    for p in [x for x in preds if not x["correct"]][:5]:
        print(f"  ✗ {p['sample_id']}  pos={p['error_position']}")
        print(f"    rep={p['replacement_token']!r}  "
              f"target={p['target_token']!r}  predicted={p['predicted_token']!r}")
        if "top5_candidates" in p:
            print(f"    top-5: {p['top5_candidates']}")


In [ ]:
# 📊 Context baselines (from saved JSON) and voice-MFC (recomputed)

print("── Context baseline results (from saved JSON) ──")
if CONTEXT_BASELINE_JSON.exists():
    cb  = json.loads(CONTEXT_BASELINE_JSON.read_text())
    N   = len(cb)
    def _h(field): return sum(1 for r in cb if r[field])
    fb_p = sum(1 for r in cb if r["prev_used_fallback"])
    fb_n = sum(1 for r in cb if r["next_used_fallback"])
    print(f"  MFC_10k    : {_h('mfc_correct'):3d}/{N} = {_h('mfc_correct')/N:.2%}")
    print(f"  prev_pitch : {_h('prev_correct'):3d}/{N} = {_h('prev_correct')/N:.2%}  (fallback {fb_p})")
    print(f"  next_pitch : {_h('next_correct'):3d}/{N} = {_h('next_correct')/N:.2%}  (fallback {fb_n})")
    print(f"  v3 (ref)   : {_h('v3_correct'):3d}/{N} = {_h('v3_correct')/N:.2%}")
    print(f"  Coverage: prev {N-fb_p}/{N}  next {N-fb_n}/{N}")
else:
    print("  context_baseline_results.json not found.")

print("\n── Voice-MFC baseline (recomputed from JSONL) ──")
vm_results, voice_mfc_map, vm_global_mfc = voice_mfc_eval(TRAIN_10K, DEV_JSONL)
vm_hits = sum(1 for r in vm_results if r["correct"])
N       = len(vm_results)
print(f"  voice_mfc  : {vm_hits:3d}/{N} = {vm_hits/N:.2%}")
print(f"  global MFC : {vm_global_mfc!r}")
print(f"\n  Per-voice MFC:")
for voice in sorted(voice_mfc_map, key=lambda x: (x is None, x or "")):
    print(f"    {voice or '(None)':10s} → {voice_mfc_map[voice]}")


In [ ]:
# 📊 Cross-run Venn: which dev examples each run gets correct
print("Cross-run correct-set overlap\n")
run_correct = {}
all_sample_ids = set()
for rn, rd in RUNS.items():
    pf = rd / "dev_predictions.json"
    if pf.exists():
        preds = json.loads(pf.read_text())
        run_correct[rn] = frozenset(p["sample_id"] for p in preds if p["correct"])
        all_sample_ids.update(p["sample_id"] for p in preds)

rns = list(run_correct.keys())
# Pairwise overlap table (diagonal = total correct for that run)
print(f"{'':>22}" + "".join(f"{r:>9}" for r in rns))
for rn_a in rns:
    row = f"{rn_a:<22}"
    for rn_b in rns:
        val = (len(run_correct[rn_a]) if rn_a == rn_b
               else len(run_correct[rn_a] & run_correct[rn_b]))
        row += f"{val:>9}"
    print(row)

N = len(all_sample_ids)
print(f"\nTotal dev: {N}")
if run_correct:
    all_correct  = frozenset.intersection(*run_correct.values())
    none_correct = all_sample_ids - frozenset.union(*run_correct.values())
    print(f"Correct in ALL runs:  {len(all_correct)}/{N}")
    print(f"Wrong  in ALL runs:   {len(none_correct)}/{N} = {len(none_correct)/N:.1%}")


## 3. Training ⚠️ TRAINS

These cells run a full training experiment. Before running:

1. Edit `EXP` in Cell 1 (name, hyperparameters, use_mask, checkpoint_by, device).
2. Ensure `EXP["output_dir"]` does not already exist.
3. Run Section 1 (setup cells) if not already done.

**MPS non-determinism:** Apple Silicon MPS produces slightly different results across
runs even with `seed=42`. Set `"device": "cpu"` in `EXP` for fully deterministic training.


In [ ]:
def run_experiment(cfg, train_ids, dev_ids, device):
    """
    ⚠️ TRAINS — Full train/eval loop.

    cfg keys:
      name, train_jsonl, use_mask, voice_conditioning, voice_dim,
      d_model, num_layers, dim_ff, checkpoint_by, seed,
      batch_size, lr, weight_decay, max_epochs, patience, time_limit, output_dir

    checkpoint_by="top1": saves on strict top-1 improvement; ties broken by lower loss.
    checkpoint_by="loss": saves on strict loss improvement.
    Patience resets only on strict improvement of the monitored metric.

    Writes to cfg['output_dir']; aborts if directory already exists.
    Returns dict with metrics, n_correct, best_epoch.
    """
    output_dir = cfg["output_dir"]
    if output_dir.exists() and any(output_dir.iterdir()):
        raise RuntimeError(
            f"Output dir not empty: {output_dir}\n"
            f"Change name or delete the directory first."
        )
    output_dir.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(cfg["seed"]); random.seed(cfg["seed"])
    rng        = random.Random(cfg["seed"])
    start_wall = time.time()
    train_id_set = set(train_ids)
    dev_id_set   = set(dev_ids)

    # Token vocab
    use_mask  = cfg["use_mask"]
    v_base    = build_vocab_from_train(DATASET_ROOT, train_ids)
    vocab_exp = add_mask_to_vocab(v_base) if use_mask else v_base
    pc_exp    = build_pitch_classes(train_ids, DATASET_ROOT)
    c2p       = {v: k for k, v in pc_exp.items()}
    (output_dir / "vocab.json").write_text(json.dumps(vocab_exp, indent=2))
    (output_dir / "pitch_classes.json").write_text(json.dumps(pc_exp, indent=2))

    # Voice conditioning
    use_voice = cfg.get("voice_conditioning", False)
    voice_dim = cfg.get("voice_dim", 32)

    # Load train first (voice_vocab=None → voice_id=0 until vocab built)
    tr, _, _ = load_all_examples(cfg["train_jsonl"], vocab_exp, pc_exp,
                                  train_id_set, "train", mask=use_mask, voice_vocab=None)
    if use_voice:
        voice_vocab = build_voice_vocab(tr)
        n_voices    = len(voice_vocab)
        for e in tr:
            e.voice_id = voice_vocab.get(e.voice_label, 0)
        (output_dir / "voice_vocab.json").write_text(json.dumps(voice_vocab, indent=2))
        print(f"Voice vocab ({n_voices}): {voice_vocab}")
    else:
        voice_vocab = None
        n_voices    = 0

    dv, _, _ = load_all_examples(DEV_JSONL, vocab_exp, pc_exp,
                                  dev_id_set, "dev", mask=use_mask, voice_vocab=voice_vocab)
    print(f"Vocab: {len(vocab_exp)}  Pitch classes: {len(pc_exp)}  "
          f"use_mask={use_mask}  voice_conditioning={use_voice}")
    print(f"Train: {len(tr)}  Dev: {len(dv)}")

    # Model config
    mcfg = {
        "vocab_size":         len(vocab_exp),
        "num_pitch_classes":  len(pc_exp),
        "d_model":            cfg["d_model"],
        "nhead":              N_HEADS,
        "num_layers":         cfg["num_layers"],
        "dim_feedforward":    cfg["dim_ff"],
        "dropout":            DROPOUT_EXPERIMENT,
        "max_len":            MAX_LEN,
        "pad_id":             vocab_exp[PAD_TOKEN],
        "voice_conditioning": use_voice,
        "n_voices":           n_voices,
        "voice_dim":          voice_dim,
    }
    if use_mask:
        mcfg["mask_token_id"] = vocab_exp[MASK_TOKEN]
        mcfg["mask_error"]    = True
    mcfg["checkpoint_by"] = cfg["checkpoint_by"]
    _nm    = {"mask_token_id", "mask_error", "checkpoint_by"}
    model  = CorrectionModel(**{k: v for k, v in mcfg.items() if k not in _nm}).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    (output_dir / "model_config.json").write_text(json.dumps(mcfg, indent=2))
    print(f"Params: {n_params:,}  checkpoint_by={cfg['checkpoint_by']}")

    loss_fn   = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                                   weight_decay=cfg["weight_decay"])

    # Epoch 0 baseline
    tl0, tt1_0, tt5_0, _ = evaluate_dataset(model, tr, cfg["batch_size"], loss_fn, device)
    dl0, dt1_0, dt5_0, _ = evaluate_dataset(model, dv, cfg["batch_size"], loss_fn, device)
    print(f"epoch 0: train {tl0:.4f}/{tt1_0:.2%}  dev {dl0:.4f}/{dt1_0:.2%}")

    metrics = [{"epoch":0, "train_loss":round(tl0,5), "train_top1":round(tt1_0,4),
                "train_top5":round(tt5_0,4), "dev_loss":round(dl0,5),
                "dev_top1":round(dt1_0,4),   "dev_top5":round(dt5_0,4),
                "elapsed_s":round(time.time()-start_wall,1)}]

    best_loss, best_top1, best_epoch, patience_count = dl0, dt1_0, 0, 0
    ckpt_by = cfg["checkpoint_by"]

    for epoch in range(1, cfg["max_epochs"] + 1):
        if time.time() - start_wall >= cfg["time_limit"]:
            print(f"Time limit reached before epoch {epoch}."); break
        es = time.time()
        tr_l, tr_t1, tr_t5 = run_train_epoch(
            model, tr, cfg["batch_size"], optimizer, loss_fn, device, rng)
        d_l, d_t1, d_t5, _ = evaluate_dataset(
            model, dv, cfg["batch_size"], loss_fn, device)
        elapsed = time.time() - start_wall
        metrics.append({"epoch":epoch, "train_loss":round(tr_l,5),
                        "train_top1":round(tr_t1,4), "train_top5":round(tr_t5,4),
                        "dev_loss":round(d_l,5), "dev_top1":round(d_t1,4),
                        "dev_top5":round(d_t5,4), "elapsed_s":round(elapsed,1)})
        print(f"  epoch {epoch:2d}  train {tr_l:.4f}/{tr_t1:.2%}  "
              f"dev {d_l:.4f}/{d_t1:.2%}  [{time.time()-es:.0f}s]")

        # Checkpoint criterion: top1 with tie-break on loss, or pure loss
        if ckpt_by == "top1":
            save_ckpt = (d_t1 > best_top1) or (d_t1 == best_top1 and d_l < best_loss)
            ptn_reset = d_t1 > best_top1      # patience on strict top1 gain only
        else:
            save_ckpt = ptn_reset = d_l < best_loss

        if save_ckpt:
            best_loss, best_top1, best_epoch = d_l, d_t1, epoch
            torch.save({"model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "epoch": epoch, "dev_loss": d_l, "dev_top1": d_t1},
                       output_dir / "best_checkpoint.pt")
            print(f"    ↑ best (by {ckpt_by})")

        if ptn_reset:
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= cfg["patience"]:
                print("  Early stopping."); break

    # Save metrics
    (output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    fields = ["epoch","train_loss","train_top1","train_top5",
              "dev_loss","dev_top1","dev_top5","elapsed_s"]
    with open(output_dir / "metrics.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(metrics)

    # Reload best checkpoint and verify
    print(f"\nReloading best checkpoint (epoch {best_epoch})...")
    ckpt = torch.load(output_dir / "best_checkpoint.pt",
                      map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    d_l_v, d_t1_v, d_t5_v, best_preds = evaluate_dataset(
        model, dv, cfg["batch_size"], loss_fn, device)
    best_row = next(m for m in metrics if m["epoch"] == best_epoch)
    assert abs(d_t1_v - best_row["dev_top1"]) < 1e-3, \
        f"Reload mismatch: {d_t1_v} vs {best_row['dev_top1']}"
    n_correct = sum(1 for e, p in zip(dv, best_preds) if e.target_class == p)
    print(f"Reload PASSED  dev top-1={n_correct}/{len(dv)} = {n_correct/len(dv):.2%}")

    # Save predictions
    logits_dev  = get_logits_for_dataset(model, dv, cfg["batch_size"], device)
    predictions = build_predictions(dv, best_preds, c2p, logits_dev)
    (output_dir / "dev_predictions.json").write_text(json.dumps(predictions, indent=2))
    saved_t1 = sum(1 for p in predictions if p["correct"])
    assert saved_t1 == n_correct, f"Saved JSON count mismatch: {saved_t1} vs {n_correct}"

    total = time.time() - start_wall
    print(f"\n{'='*60}")
    print(f" DONE: {n_correct}/{len(dv)} = {n_correct/len(dv):.2%} dev top-1")
    print(f" Best epoch: {best_epoch}  Total: {total:.0f}s")
    print(f" Artifacts: {output_dir}")
    print(f"{'='*60}")
    return {"metrics": metrics, "n_correct": n_correct, "best_epoch": best_epoch,
            "dev_top1": d_t1_v, "dev_top5": d_t5_v, "n_params": n_params,
            "elapsed_s": total}


In [ ]:
# ⚠️ TRAINS — Run this cell to launch a new experiment.
# 1. Edit EXP in Cell 1 (name, hyperparams, use_mask, checkpoint_by, device).
# 2. EXP["output_dir"] must not already exist.
# 3. Section 1 (setup) must have run (train_ids, dev_ids, DEVICE must be defined).
#
# After training, add the run to RUNS with:
#   RUNS[EXP["name"]] = EXP["output_dir"]

result = run_experiment(EXP, train_ids, dev_ids, DEVICE)
print(f"\nTo add to RUNS: RUNS[{EXP['name']!r}] = RUNS_DIR / {EXP['name']!r}")


## 3b. v5 Experiment — Explicit Voice Conditioning

Control (v4 replication) vs voice-conditioned model, trained with identical settings.
Both checkpoint by highest dev top-1; ties broken by lower dev loss.

In [ ]:
# ⚙️ v5 experiment configs — identical settings, only voice_conditioning differs
_SHARED = dict(
    train_jsonl      = TRAIN_10K,
    use_mask         = True,
    voice_dim        = 32,
    d_model          = 128,
    num_layers       = 3,
    dim_ff           = 512,
    checkpoint_by    = "top1",   # saved on strict dev top-1 gain; ties → lower loss
    seed             = 42,
    batch_size       = 32,
    lr               = 1e-3,
    weight_decay     = 0.01,
    max_epochs       = 30,
    patience         = 5,
    time_limit       = 870,      # 14.5 min per run ≤ 30 min combined
    device           = None,     # None = auto-detect MPS/CUDA/CPU
)

EXP_V4_CTRL = {**_SHARED, "name": "v5_v4_control",         "voice_conditioning": False}
EXP_V5_VOICE = {**_SHARED, "name": "v5_voice_conditioned", "voice_conditioning": True}

for cfg in (EXP_V4_CTRL, EXP_V5_VOICE):
    cfg["output_dir"] = RUNS_DIR / cfg["name"]
    print(f"{cfg['name']}: voice_conditioning={cfg['voice_conditioning']}  → {cfg['output_dir']}")


In [ ]:
# ⚠️ TRAINS — v4 control (no voice conditioning)
# Run Section 1 (cells 11–12) before this cell.
_dev  = (torch.device(EXP_V4_CTRL["device"]) if EXP_V4_CTRL["device"] else
         torch.device("cuda" if torch.cuda.is_available() else
                      "mps"  if torch.backends.mps.is_available() else "cpu"))
result_ctrl = run_experiment(EXP_V4_CTRL, train_ids, dev_ids, _dev)


In [ ]:
# ⚠️ TRAINS — v5 voice-conditioned
_dev  = (torch.device(EXP_V5_VOICE["device"]) if EXP_V5_VOICE["device"] else
         torch.device("cuda" if torch.cuda.is_available() else
                      "mps"  if torch.backends.mps.is_available() else "cpu"))
result_voice = run_experiment(EXP_V5_VOICE, train_ids, dev_ids, _dev)


In [ ]:
# 📊 v4 control vs v5 voice-conditioned comparison
# Reads from saved dev_predictions.json; requires both training cells to have run.

_ctrl_dir  = RUNS_DIR / "v5_v4_control"
_voice_dir = RUNS_DIR / "v5_voice_conditioned"

def _load_preds(run_dir):
    pf = run_dir / "dev_predictions.json"
    mc = run_dir / "model_config.json"
    ck = run_dir / "best_checkpoint.pt"
    if not pf.exists():
        raise FileNotFoundError(f"No predictions at {pf}")
    preds  = json.loads(pf.read_text())
    n_corr = sum(1 for p in preds if p["correct"])
    n_tot  = len(preds)
    n_prms = json.loads(mc.read_text()) if mc.exists() else {}
    ckpt_ep = (torch.load(ck, map_location="cpu", weights_only=False)["epoch"]
               if ck.exists() else "?")
    metrics = json.loads((run_dir / "metrics.json").read_text())
    elapsed = metrics[-1]["elapsed_s"]
    return preds, n_corr, n_tot, ckpt_ep, elapsed

import torch as _t2

p_ctrl,  nc_c, nt, ep_c, el_c = _load_preds(_ctrl_dir)
p_voice, nc_v, _,  ep_v, el_v = _load_preds(_voice_dir)

mc_c  = json.loads((_ctrl_dir  / "model_config.json").read_text())
mc_v  = json.loads((_voice_dir / "model_config.json").read_text())

# Reconstruct models to count params (config already has them; just use stored n_params)
# Use metrics file n_params if available, else compute from config
def _n_params(mc):
    """Estimate total params from model config dict."""
    vc   = mc.get("voice_conditioning", False)
    vd   = mc.get("voice_dim", 32)
    nv   = mc.get("n_voices", 0)
    dm   = mc["d_model"]
    vs   = mc["vocab_size"]
    pc   = mc["num_pitch_classes"]
    ml   = mc["max_len"]
    nl   = mc["num_layers"]
    df   = mc["dim_feedforward"]
    nh   = mc["nhead"]
    emb  = vs * dm + ml * dm
    lay  = nl * (3*dm*dm + 3*dm + dm*dm + dm + dm*df + df + df*dm + dm + 2*2*dm)
    cin  = dm + vd if vc else dm
    clf  = cin * pc + pc
    vemb = (nv + 1) * vd if vc else 0
    return emb + lay + clf + vemb

np_c = _n_params(mc_c)
np_v = _n_params(mc_v)

# Dev top-5
def _top5(preds):
    return sum(1 for p in preds if any(c == p["target_token"]
               for c in p.get("top5_candidates", []))) / len(preds)

t5_c = _top5(p_ctrl)
t5_v = _top5(p_voice)

# ── Summary table ────────────────────────────────────────────────────────────
hdr = f"{'':26} {'v4_control':>16} {'v5_voice':>16} {'Δ':>8}"
sep = "─" * len(hdr)
print(hdr); print(sep)
print(f"{'Dev top-1':<26} {nc_c}/{nt}={nc_c/nt:.2%}  {nc_v}/{nt}={nc_v/nt:.2%}  "
      f"  {(nc_v-nc_c)/nt:+.2%}")
print(f"{'Dev top-5':<26} {t5_c:>16.2%} {t5_v:>16.2%} {t5_v-t5_c:>+8.2%}")
print(f"{'Params (est.)':<26} {np_c:>16,} {np_v:>16,} {np_v-np_c:>+8,}")
print(f"{'Best epoch':<26} {ep_c:>16} {ep_v:>16}")
print(f"{'Train wall time (s)':<26} {el_c:>16.0f} {el_v:>16.0f}")

# ── Per-voice breakdown ───────────────────────────────────────────────────────
print(f"\n{'─'*70}")
print(f"Per-voice accuracy")
print(f"{'─'*70}")

voice_data = {}
for src, preds in [("ctrl", p_ctrl), ("voice", p_voice)]:
    for p in preds:
        vl = p.get("voice_label") or "(None)"
        if vl not in voice_data:
            voice_data[vl] = {"n": 0, "ctrl": 0, "voice": 0}
        voice_data[vl]["n"]    += 1  # counted from ctrl side (same examples)
        voice_data[vl][src]    += int(p["correct"])

# Remove double-counting of n (both preds have same examples)
for vl in voice_data:
    voice_data[vl]["n"] //= 2   # each label was counted from both preds loops

# Re-count from ctrl only
voice_data2 = {}
for p in p_ctrl:
    vl = p.get("voice_label") or "(None)"
    if vl not in voice_data2:
        voice_data2[vl] = {"n": 0, "ctrl": 0}
    voice_data2[vl]["n"] += 1
    voice_data2[vl]["ctrl"] += int(p["correct"])
for p in p_voice:
    vl = p.get("voice_label") or "(None)"
    voice_data2.setdefault(vl, {"n": 0, "ctrl": 0})["__voice"] = voice_data2.setdefault(vl, {}).get("__voice", 0) + int(p["correct"])

# Clean approach
vd = {}
for p in p_ctrl:
    vl = p.get("voice_label") or "(None)"
    vd.setdefault(vl, {"n":0,"c":0,"v":0})["n"] += 1
    vd[vl]["c"] += int(p["correct"])
for p in p_voice:
    vl = p.get("voice_label") or "(None)"
    vd.setdefault(vl, {"n":0,"c":0,"v":0})["v"] += int(p["correct"])

vhdr = f"{'Voice':<12} {'N':>5} {'ctrl top-1':>12} {'voice top-1':>13} {'Δ':>8}"
print(vhdr)
print("─" * len(vhdr))
for vl, d in sorted(vd.items(), key=lambda x: -x[1]["n"]):
    n = d["n"]
    if n == 0: continue
    ac = d["c"] / n; av = d["v"] / n
    print(f"{vl:<12} {n:>5} {d['c']:>4}/{n}={ac:.1%} {d['v']:>4}/{n}={av:.1%} {av-ac:>+8.1%}")

# ── Venn: common/unique correct ───────────────────────────────────────────────
sid_ctrl  = {p["sample_id"] for p in p_ctrl  if p["correct"]}
sid_voice = {p["sample_id"] for p in p_voice if p["correct"]}
sid_all   = {p["sample_id"] for p in p_ctrl}
both     = sid_ctrl & sid_voice
only_c   = sid_ctrl  - sid_voice
only_v   = sid_voice - sid_ctrl
neither  = sid_all - sid_ctrl - sid_voice

print(f"\n{'─'*50}")
print(f"Prediction overlap (N={len(sid_all)})")
print(f"{'─'*50}")
print(f"  Both correct    : {len(both):3d}/{len(sid_all)} = {len(both)/len(sid_all):.1%}")
print(f"  ctrl-only       : {len(only_c):3d}/{len(sid_all)} = {len(only_c)/len(sid_all):.1%}")
print(f"  voice-only      : {len(only_v):3d}/{len(sid_all)} = {len(only_v)/len(sid_all):.1%}")
print(f"  Neither correct : {len(neither):3d}/{len(sid_all)} = {len(neither)/len(sid_all):.1%}")

# ── Octave error comparison ───────────────────────────────────────────────────
def _octave_errors(preds):
    wrong = [p for p in preds if not p["correct"]]
    if not wrong: return 0, len(preds) - len(wrong), len(preds)
    oct_ok = sum(1 for p in wrong
                 if p["target_token"][1] == p["predicted_token"][1])
    return oct_ok, len(preds) - len(wrong), len(preds)

oc_c, nc_co, nt2 = _octave_errors(p_ctrl)
oc_v, nc_vo, _   = _octave_errors(p_voice)
n_wrong_c = nt2 - nc_co
n_wrong_v = nt2 - nc_vo

print(f"\n{'─'*50}")
print(f"Octave errors among wrong predictions")
print(f"{'─'*50}")
if n_wrong_c:
    print(f"  ctrl  wrong={n_wrong_c}  correct octave={oc_c}/{n_wrong_c}={oc_c/n_wrong_c:.1%}")
if n_wrong_v:
    print(f"  voice wrong={n_wrong_v}  correct octave={oc_v}/{n_wrong_v}={oc_v/n_wrong_v:.1%}")

print(f"\nArtifacts:")
print(f"  ctrl  → {_ctrl_dir}")
print(f"  voice → {_voice_dir}")


## 3c. v6 Experiment — Larger Voice-Conditioned Model

Same as v5 (MASK + voice conditioning), only architecture size changes:
`d_model 128→256`, `num_layers 3→4`, `dim_feedforward 512→1024`.

In [ ]:
# ⚙️ v6 experiment config
# Identical to v5_voice_conditioned except architecture size.
EXP_V6 = {
    "name":               "v6_voice_large",
    "train_jsonl":        TRAIN_10K,
    "use_mask":           True,
    "voice_conditioning": True,
    "voice_dim":          32,      # unchanged from v5
    "d_model":            256,     # ↑ 128 → 256
    "num_layers":         4,       # ↑ 3   → 4
    "dim_ff":             1024,    # ↑ 512 → 1024
    "checkpoint_by":      "top1",
    "seed":               42,
    "batch_size":         32,
    "lr":                 1e-3,
    "weight_decay":       0.01,
    "max_epochs":         30,
    "patience":           5,
    "time_limit":         1800,
    "device":             None,
}
EXP_V6["output_dir"] = RUNS_DIR / EXP_V6["name"]
print(f"v6: d_model={EXP_V6['d_model']} num_layers={EXP_V6['num_layers']} "
      f"dim_ff={EXP_V6['dim_ff']}  → {EXP_V6['output_dir']}")


In [ ]:
# ⚠️ TRAINS — v6 larger voice-conditioned model
# Requires Section 1 (cells 11–12) to have run first.
_dev6 = (torch.device(EXP_V6["device"]) if EXP_V6["device"] else
         torch.device("cuda" if torch.cuda.is_available() else
                      "mps"  if torch.backends.mps.is_available() else "cpu"))
result_v6 = run_experiment(EXP_V6, train_ids, dev_ids, _dev6)


In [ ]:
# 📊 v5 vs v6 comparison
# Loads saved dev_predictions.json from both runs; verifies counts from artifacts.

import torch as _t3

_v5_dir = RUNS_DIR / "v5_voice_conditioned"
_v6_dir = RUNS_DIR / "v6_voice_large"

def _load_run(run_dir):
    preds   = json.loads((run_dir / "dev_predictions.json").read_text())
    mc      = json.loads((run_dir / "model_config.json").read_text())
    metrics = json.loads((run_dir / "metrics.json").read_text())
    ck      = _t3.load(run_dir / "best_checkpoint.pt",
                       map_location="cpu", weights_only=False)
    n_corr  = sum(1 for p in preds if p["correct"])
    t5      = sum(1 for p in preds
                  if p["target_token"] in p.get("top5_candidates", []))
    return {
        "preds":    preds,
        "n_corr":   n_corr,
        "n_tot":    len(preds),
        "t5":       t5,
        "ckpt_ep":  ck["epoch"],
        "elapsed":  metrics[-1]["elapsed_s"],
        "n_ep":     len(metrics) - 1,
        "mc":       mc,
    }

r5 = _load_run(_v5_dir)
r6 = _load_run(_v6_dir)
N  = r5["n_tot"]

# Verify v5 reference (83/499 top-1, 261/499 top-5)
assert r5["n_corr"] == 83,  f"v5 top-1 mismatch: {r5['n_corr']}"
assert r5["t5"]     == 261, f"v5 top-5 mismatch: {r5['t5']}"

# ── Summary table ─────────────────────────────────────────────────────────────
def _sep(w=66): return "─" * w
hdr = f"{'':28} {'v5_voice (small)':>18} {'v6_voice_large':>18} {'Δ':>7}"
print(hdr); print(_sep())
nc5, nc6 = r5["n_corr"], r6["n_corr"]
t55, t56 = r5["t5"],     r6["t5"]
print(f"{'Dev top-1':<28} {nc5}/{N}={nc5/N:.2%}  {nc6}/{N}={nc6/N:.2%}  {(nc6-nc5)/N:>+.2%}")
print(f"{'Dev top-5':<28} {t55}/{N}={t55/N:.2%}  {t56}/{N}={t56/N:.2%}  {(t56-t55)/N:>+.2%}")
print(f"{'Best epoch / total':<28} {r5['ckpt_ep']}/{r5['n_ep']:>18} "
      f"{r6['ckpt_ep']}/{r6['n_ep']:>18}")
print(f"{'Wall time (s)':<28} {r5['elapsed']:>18.0f} {r6['elapsed']:>18.0f}")

for key, label in [("d_model","d_model"), ("num_layers","num_layers"),
                    ("dim_feedforward","dim_ff")]:
    print(f"{label:<28} {r5['mc'].get(key,'?'):>18} {r6['mc'].get(key,'?'):>18}")

# Param count: sum all tensors in best_checkpoint.pt model_state_dict
def _param_count(run_dir):
    sd = _t3.load(run_dir / "best_checkpoint.pt",
                  map_location="cpu", weights_only=False)["model_state_dict"]
    return sum(v.numel() for v in sd.values())

np5 = _param_count(_v5_dir)
np6 = _param_count(_v6_dir)
print(f"{'Params':<28} {np5:>18,} {np6:>18,}  {np6-np5:>+7,}")

# ── Per-voice breakdown ────────────────────────────────────────────────────────
print(f"\n{_sep()}")
print("Per-voice accuracy  (all 499 examples, UNKNOWN included)")
print(_sep())

vd = {}
for p in r5["preds"]:
    vl = p.get("voice_label") or "(None/UNKNOWN)"
    vd.setdefault(vl, {"n":0,"v5":0,"v6":0})["n"]  += 1
    vd[vl]["v5"] += int(p["correct"])
for p in r6["preds"]:
    vl = p.get("voice_label") or "(None/UNKNOWN)"
    vd.setdefault(vl, {"n":0,"v5":0,"v6":0})["v6"] += int(p["correct"])

vhdr = f"{'Voice':<18} {'N':>5}  {'v5 top-1':>10} {'v6 top-1':>11}  {'Δ':>7}"
print(vhdr); print("─"*len(vhdr))
for vl, d in sorted(vd.items(), key=lambda x: -x[1]["n"]):
    n = d["n"]
    a5 = d["v5"]/n; a6 = d["v6"]/n
    print(f"{vl:<18} {n:>5}  {d['v5']:>3}/{n}={a5:.1%}  "
          f"{d['v6']:>3}/{n}={a6:.1%}  {a6-a5:>+7.1%}")

# ── Venn overlap ───────────────────────────────────────────────────────────────
sid5  = {p["sample_id"] for p in r5["preds"] if p["correct"]}
sid6  = {p["sample_id"] for p in r6["preds"] if p["correct"]}
sida  = {p["sample_id"] for p in r5["preds"]}
both   = sid5 & sid6
only5  = sid5 - sid6
only6  = sid6 - sid5
neither = sida - sid5 - sid6

print(f"\n{_sep(50)}")
print(f"Prediction overlap  (N={N})")
print(_sep(50))
print(f"  Both correct    : {len(both):3d}/{N} = {len(both)/N:.1%}")
print(f"  v5-only correct : {len(only5):3d}/{N} = {len(only5)/N:.1%}")
print(f"  v6-only correct : {len(only6):3d}/{N} = {len(only6)/N:.1%}")
print(f"  Neither correct : {len(neither):3d}/{N} = {len(neither)/N:.1%}")

# ── Octave accuracy (all examples) ────────────────────────────────────────────
def _oct_acc(preds):
    """Octave-correct rate over all wrong predictions."""
    wrong = [p for p in preds if not p["correct"]]
    if not wrong: return 0.0, 0
    ok = sum(1 for p in wrong if p["target_token"][1] == p["predicted_token"][1])
    return ok / len(wrong), len(wrong)

oa5, nw5 = _oct_acc(r5["preds"])
oa6, nw6 = _oct_acc(r6["preds"])
print(f"\n{_sep(50)}")
print(f"Octave accuracy among wrong predictions")
print(_sep(50))
print(f"  v5 wrong={nw5}  correct octave: {oa5:.1%}")
print(f"  v6 wrong={nw6}  correct octave: {oa6:.1%}  Δ={oa6-oa5:+.1%}")

print(f"\nArtifacts:")
print(f"  v5 → {_v5_dir}")
print(f"  v6 → {_v6_dir}")


### v6 underfitting analysis

**Diagnostic signal — train accuracy lower than dev accuracy at best checkpoint:**

| Run | train top-1 | dev top-1 | train loss | dev loss |
|-----|-------------|-----------|------------|----------|
| v5_voice (ep 10) | 17.73% | **16.63%** | 2.56 | 2.79 |
| v6_voice_large (ep 14) | 8.03% | **11.22%** | 3.03 | 3.03 |

v5 shows the expected pattern: train > dev, loss diverging. v6 inverts it — the model predicts the training examples *less accurately* than the held-out ones. Train loss and dev loss are nearly identical (3.03 vs 3.03) and barely decreased from epoch 0 (~3.95), indicating the optimizer failed to find a useful minimum.

**Root cause: parameter-to-data ratio.**
v6 has 3,287,252 parameters trained on 9,998 examples (329 params/example). v5 has 659,924 on the same data (66 params/example). At lr=1e-3 with no warmup, the gradient signal from each batch is too sparse to steer a 3.3M-param model reliably. The model converges to a nearly random-looking minimum rather than overfitting.

**Per-voice regression is uniform**, not voice-specific: voice:1 fell from 19.2% to 11.4% (−7.7%), voice:5 from 13.3% to 11.1% (−2.2%), voice:6 from 15.8% to 5.3% (−10.5%). No voice benefited from the larger architecture. The octave-correct rate among wrong predictions also fell (49.5% → 44.2%), confirming the model learned weaker representations overall.

**Experiment conclusions:**

1. **Voice conditioning is the dominant signal** (+5.21% absolute over unvoiced control at same architecture). This result is robust.
2. **Architecture size does not help at 10K examples.** Going from 660K to 3.3M params regresses by −5.41%. The current bottleneck is data quantity, not model capacity.
3. **v5 (128d · 3L · 512ff + voice, 660K params) remains the best model**: 83/499 = 16.63% dev top-1, 261/499 = 52.30% dev top-5.

**Recommended next experiments (in order of expected yield):**

- **v7a — more training data**: generate a second 10K split (different seed) and train v5-architecture on 20K. Data quantity is the identified bottleneck; architecture is already well-matched.
- **v7b — lr schedule on v6**: train v6 with linear warmup (3 epochs) + cosine decay to test whether the large model can learn with a more careful schedule. Requires no new data.
- **v7c — voice embedding ablation**: replace the continuous learned voice embedding with a one-hot feature passed directly to the classifier head (no backprop through it). Isolates whether the voice signal is used for representation or just routing.


## 3d. v7 Experiment — Dynamic Masked-Pitch Training

**Hypothesis:** Randomising the masked position each epoch (dynamic corruption, MDTK-style)
teaches broader pitch-context representations than always masking the same fixed position.

**Design**:
- Architecture: identical to v5 (128d·3L·512ff, voice conditioning, seed=42)
- Train set: same 9,998 examples; correct tokens from LMX source files
- Each epoch: one eligible pitch position per example, selected by `Random(epoch×9999991 + jsonl_idx)`
- Eligible = any position where the token is in `pitch_classes` (52 classes)
- [MASK] applied at selected position; target = original pitch at that position
- Dev: same 499 fixed examples (masked at original error positions)
- Budget: 4,695 optimizer steps = v5 total (15 epochs × 313 steps)
- Checkpoint criterion: highest dev top-1; ties broken by lower dev loss; patience=5

**Reference (MDTK):** Cífka et al., "Groove2Groove" / dynamic corruption in discrete-token
masked prediction; each epoch sees a fresh corruption sample.


In [ ]:
# ⚡ v7 pre-processing: cache LMX source files and eligible pitch positions (run once)
# Requires Section 1 (cells 11–12) to be run first so train_ids, vocab, pitch_classes exist.
# Re-run is safe: just re-builds the cache in memory.

_v5_dir = RUNS_DIR / "v5_voice_conditioned"

# Load v5 artifacts (vocab identical to what Section 1 built; reload for explicit traceability)
_v7_vocab       = json.loads((_v5_dir / "vocab.json").read_text())
_v7_pc          = json.loads((_v5_dir / "pitch_classes.json").read_text())
_v7_voice_vocab = json.loads((_v5_dir / "voice_vocab.json").read_text())
assert _v7_vocab == vocab,          "v7 vocab differs from Section 1 vocab — re-run Section 1"
assert _v7_pc    == pitch_classes,  "v7 pitch_classes differ — re-run Section 1"
print(f"v7 artifacts match Section 1  (vocab={len(_v7_vocab)}, PC={len(_v7_pc)}, voices={len(_v7_voice_vocab)})")

# Verify no dev/train score ID overlap (redundant with Section 1 but explicit)
_tr_score_ids  = {sid.split('/')[1] for sid in train_ids}
_dev_score_ids = {sid.split('/')[1] for sid in dev_ids}
_overlap = _tr_score_ids & _dev_score_ids
assert not _overlap, f"Dev/train score ID overlap: {_overlap}"
print(f"Score ID overlap: CLEAN  (train={len(_tr_score_ids)}, dev={len(_dev_score_ids)})")

# Cache LMX source files: sample_id → correct token list
# Eligible positions: sample_id → [(src_pos, pitch_tok), ...]
print("Caching LMX source files...")
_v7_lmx_tokens   = {}
_v7_eligible_pos = {}
_v7_missing      = []

for _e in train_examples:
    _sid = _e.sample_id
    if _sid in _v7_lmx_tokens:
        continue
    _p = DATASET_ROOT / f"{_sid}.lmx"
    if not _p.exists():
        _v7_missing.append(_sid); continue
    _toks = _p.read_text().split()
    _v7_lmx_tokens[_sid]   = _toks
    _v7_eligible_pos[_sid] = [(i, t) for i, t in enumerate(_toks) if t in _v7_pc]

if _v7_missing:
    print(f"WARNING: {len(_v7_missing)} LMX files not found")
else:
    print(f"All {len(_v7_lmx_tokens)} LMX source files loaded")

_total_elig = sum(len(v) for v in _v7_eligible_pos.values())
_min_elig   = min(len(v) for v in _v7_eligible_pos.values())
_max_elig   = max(len(v) for v in _v7_eligible_pos.values())
print(f"Eligible positions: total={_total_elig:,}  "
      f"per-sample min={_min_elig}  max={_max_elig}  mean={_total_elig/len(_v7_eligible_pos):.1f}")

# Smoke test: 5 random samples
import random as _rnd
_srng = _rnd.Random(0)
for _sid in _srng.sample(list(_v7_lmx_tokens.keys()), 5):
    _toks  = _v7_lmx_tokens[_sid]
    _elig  = _v7_eligible_pos[_sid]
    _pos, _ptok = _elig[len(_elig) // 2]
    assert _toks[_pos] == _ptok
    _win, _wpos, _ = window_tokens(_toks, _pos, _ptok)
    _vlabel = build_voice_map(_win).get(_wpos)
    _win_m  = list(_win); _win_m[_wpos] = MASK_TOKEN
    assert _win_m[_wpos] == MASK_TOKEN
    assert _v7_pc.get(_ptok) is not None
    print(f"  {_sid}: pos={_pos}  pitch={_ptok}  voice={_vlabel}  win={len(_win)}")
print("Smoke test PASSED — LMX cache ready for v7 training")


In [ ]:
# ⚙️ v7 dynamic masking config
# Budget: v5 total optimizer steps = 15 training epochs × ceil(9998/32)=313 steps = 4,695

_V7_STEP_BUDGET = 15 * 313  # = 4695 (matches v5 total; verified from metrics.csv)

EXP_V7_DYNAMIC = {
    "name":              "v7_dynamic_mask",
    "d_model":           128,     # same as v5
    "num_layers":        3,       # same as v5
    "dim_ff":            512,     # same as v5
    "voice_conditioning": True,
    "voice_dim":         32,
    "checkpoint_by":     "top1",
    "seed":              42,
    "batch_size":        32,
    "lr":                1e-3,
    "weight_decay":      0.01,
    "max_epochs":        15,      # ceil(4695 / 313) = 15
    "patience":          5,
    "time_limit":        600,     # 10 min wall-time limit
    "step_budget":       _V7_STEP_BUDGET,
    "output_dir":        RUNS_DIR / "v7_dynamic_mask",
}
print(f"v7 config: step_budget={_V7_STEP_BUDGET}  max_epochs={EXP_V7_DYNAMIC['max_epochs']}  "
      f"arch={EXP_V7_DYNAMIC['d_model']}d/{EXP_V7_DYNAMIC['num_layers']}L/{EXP_V7_DYNAMIC['dim_ff']}ff")
print(f"Output: {EXP_V7_DYNAMIC['output_dir']}")


In [ ]:
# Helper: dynamic example builder
# Each call produces a fresh set of examples with different masked positions.

def _make_dynamic_epoch(epoch_num, base_examples, lmx_tokens, eligible_pos,
                         vocab, pitch_classes, voice_vocab):
    """
    Build training examples for epoch_num using deterministic dynamic masking.

    Position selection: random.Random(epoch_num * 9999991 + jsonl_idx).randrange(len(positions))
    Ensures each (epoch, sample) pair always maps to the same position → reproducible.
    'replacement_token' is set to the masked pitch (= target) to satisfy CorrectionExample.

    Returns (examples_list, seen_positions_set).
    """
    unk_id, pad_id = vocab[UNK_TOKEN], vocab[PAD_TOKEN]
    examples, seen_pos = [], set()
    for idx, base_ex in enumerate(base_examples):
        sid       = base_ex.sample_id
        positions = eligible_pos.get(sid, [])
        if not positions:
            continue
        pos_idx          = random.Random(epoch_num * 9999991 + idx).randrange(len(positions))
        src_pos, pitch_tok = positions[pos_idx]
        src_toks         = lmx_tokens[sid]
        window, w_pos, w_lo = window_tokens(src_toks, src_pos, pitch_tok)
        voice_label      = build_voice_map(window).get(w_pos)
        voice_id         = voice_vocab.get(voice_label, 0)
        window           = list(window)
        window[w_pos]    = MASK_TOKEN
        ids     = [vocab.get(t, unk_id) for t in window]
        pad_len = MAX_LEN - len(ids)
        examples.append(CorrectionExample(
            sample_id         = sid,
            token_ids         = ids + [pad_id] * pad_len,
            padding_mask      = [False] * len(ids) + [True] * pad_len,
            window_error_pos  = w_pos,
            target_class      = pitch_classes[pitch_tok],
            window_lo         = w_lo,
            target_token      = pitch_tok,
            replacement_token = pitch_tok,   # dynamic: masking the correct token
            voice_label       = voice_label,
            voice_id          = voice_id,
        ))
        seen_pos.add((sid, src_pos))
    return examples, seen_pos


def run_experiment_dynamic(cfg, base_train, dev_examples,
                            lmx_tokens, eligible_pos, vocab, pitch_classes,
                            voice_vocab, device):
    """
    ⚠️ TRAINS — Dynamic masked-pitch training loop.

    Each epoch a fresh set of examples is generated via _make_dynamic_epoch.
    Train metrics are online (from training batches); dev is fully evaluated once per epoch.
    Budget enforced by both max_epochs and step_budget.
    """
    output_dir = cfg["output_dir"]
    if output_dir.exists() and any(output_dir.iterdir()):
        raise RuntimeError(f"Output dir not empty: {output_dir}")
    output_dir.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(cfg["seed"]); random.seed(cfg["seed"])
    rng        = random.Random(cfg["seed"])
    start_wall = time.time()
    n_voices   = len(voice_vocab)
    c2p        = {v: k for k, v in pitch_classes.items()}

    (output_dir / "vocab.json").write_text(json.dumps(vocab, indent=2))
    (output_dir / "pitch_classes.json").write_text(json.dumps(pitch_classes, indent=2))
    (output_dir / "voice_vocab.json").write_text(json.dumps(voice_vocab, indent=2))

    mcfg = {
        "vocab_size":         len(vocab),
        "num_pitch_classes":  len(pitch_classes),
        "d_model":            cfg["d_model"],
        "nhead":              N_HEADS,
        "num_layers":         cfg["num_layers"],
        "dim_feedforward":    cfg["dim_ff"],
        "dropout":            DROPOUT_EXPERIMENT,
        "max_len":            MAX_LEN,
        "pad_id":             vocab[PAD_TOKEN],
        "voice_conditioning": True,
        "n_voices":           n_voices,
        "voice_dim":          cfg["voice_dim"],
        "mask_token_id":      vocab[MASK_TOKEN],
        "mask_error":         True,
        "checkpoint_by":      cfg["checkpoint_by"],
    }
    _nm    = {"mask_token_id", "mask_error", "checkpoint_by"}
    model  = CorrectionModel(**{k: v for k, v in mcfg.items() if k not in _nm}).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    (output_dir / "model_config.json").write_text(json.dumps(mcfg, indent=2))
    print(f"Params: {n_params:,}  checkpoint_by={cfg['checkpoint_by']}")

    loss_fn   = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                                   weight_decay=cfg["weight_decay"])

    # Epoch 0: dev baseline only (no train pass)
    dl0, dt1_0, dt5_0, _ = evaluate_dataset(model, dev_examples, cfg["batch_size"], loss_fn, device)
    print(f"epoch  0 (baseline): dev {dl0:.4f}/{dt1_0:.2%}")

    metrics = [{
        "epoch": 0, "train_loss": None, "train_top1": None, "train_top5": None,
        "dev_loss": round(dl0,5), "dev_top1": round(dt1_0,4), "dev_top5": round(dt5_0,4),
        "elapsed_s": round(time.time()-start_wall,1),
        "total_steps": 0, "distinct_positions_epoch": 0,
    }]

    best_loss=dl0; best_top1=dt1_0; best_epoch=0; patience_count=0
    total_steps = 0
    all_seen_pos = set()

    for epoch in range(1, cfg["max_epochs"] + 1):
        if time.time() - start_wall >= cfg["time_limit"]:
            print(f"Time limit reached before epoch {epoch}."); break
        if total_steps >= cfg["step_budget"]:
            print(f"Step budget ({cfg['step_budget']}) reached."); break

        es = time.time()
        tr_epoch, seen_pos = _make_dynamic_epoch(
            epoch, base_train, lmx_tokens, eligible_pos, vocab, pitch_classes, voice_vocab)
        all_seen_pos |= seen_pos
        steps_epoch = (len(tr_epoch) + cfg["batch_size"] - 1) // cfg["batch_size"]

        # Train (online metrics — not a separate eval pass over train)
        tr_l, tr_t1, tr_t5 = run_train_epoch(
            model, tr_epoch, cfg["batch_size"], optimizer, loss_fn, device, rng)
        total_steps += steps_epoch

        d_l, d_t1, d_t5, _ = evaluate_dataset(
            model, dev_examples, cfg["batch_size"], loss_fn, device)
        elapsed = time.time() - start_wall
        metrics.append({
            "epoch": epoch, "train_loss": round(tr_l,5),
            "train_top1": round(tr_t1,4), "train_top5": round(tr_t5,4),
            "dev_loss": round(d_l,5), "dev_top1": round(d_t1,4), "dev_top5": round(d_t5,4),
            "elapsed_s": round(elapsed,1), "total_steps": total_steps,
            "distinct_positions_epoch": len(seen_pos),
        })
        print(f"  epoch {epoch:2d}  train {tr_l:.4f}/{tr_t1:.2%}  "
              f"dev {d_l:.4f}/{d_t1:.2%}  pos={len(seen_pos)}  "
              f"steps={total_steps}  [{time.time()-es:.0f}s]")

        save_ckpt = (d_t1 > best_top1) or (d_t1 == best_top1 and d_l < best_loss)
        ptn_reset = d_t1 > best_top1
        if save_ckpt:
            best_loss, best_top1, best_epoch = d_l, d_t1, epoch
            torch.save({"model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "epoch": epoch, "dev_loss": d_l, "dev_top1": d_t1},
                       output_dir / "best_checkpoint.pt")
            print(f"    ↑ best")
        if ptn_reset:
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= cfg["patience"]:
                print("  Early stopping."); break

    (output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    fields = ["epoch","train_loss","train_top1","train_top5","dev_loss","dev_top1","dev_top5",
              "elapsed_s","total_steps","distinct_positions_epoch"]
    with open(output_dir / "metrics.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(metrics)

    # Coverage report
    total_elig_all = sum(len(v) for v in eligible_pos.values())
    cov_info = {
        "unique_positions_seen":    len(all_seen_pos),
        "total_eligible_positions": total_elig_all,
        "coverage_pct":             round(len(all_seen_pos) / total_elig_all * 100, 2),
        "training_epochs":          epoch,
        "total_optimizer_steps":    total_steps,
        "v5_step_budget":           cfg["step_budget"],
        "budget_used_pct":          round(total_steps / cfg["step_budget"] * 100, 1),
    }
    (output_dir / "coverage.json").write_text(json.dumps(cov_info, indent=2))
    print(f"\nCoverage: {len(all_seen_pos):,}/{total_elig_all:,} = {cov_info['coverage_pct']:.1f}%")
    print(f"Steps: {total_steps}/{cfg['step_budget']} = {cov_info['budget_used_pct']:.0f}%")

    # Reload and verify
    print(f"\nReloading best checkpoint (epoch {best_epoch})...")
    ckpt = torch.load(output_dir / "best_checkpoint.pt",
                      map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    d_l_v, d_t1_v, d_t5_v, best_preds = evaluate_dataset(
        model, dev_examples, cfg["batch_size"], loss_fn, device)
    best_row = next(m for m in metrics if m["epoch"] == best_epoch)
    assert abs(d_t1_v - best_row["dev_top1"]) < 1e-3,         f"Reload mismatch: {d_t1_v:.4f} vs {best_row['dev_top1']:.4f}"
    n_correct = sum(1 for e, p in zip(dev_examples, best_preds) if e.target_class == p)
    print(f"Reload PASSED  dev top-1={n_correct}/499 = {n_correct/499:.2%}")

    logits_dev  = get_logits_for_dataset(model, dev_examples, cfg["batch_size"], device)
    predictions = build_predictions(dev_examples, best_preds, c2p, logits_dev)
    (output_dir / "dev_predictions.json").write_text(json.dumps(predictions, indent=2))
    saved_t1 = sum(1 for p in predictions if p["correct"])
    assert saved_t1 == n_correct

    total = time.time() - start_wall
    print(f"\n{'='*60}")
    print(f" DONE: {n_correct}/499 = {n_correct/499:.2%} dev top-1")
    top5_count = sum(1 for p in predictions if p['target_token'] in p['top5_candidates'])
    print(f"       dev top-5 = {top5_count}/499 = {top5_count/499:.2%}")
    print(f" Best epoch: {best_epoch}  Total: {total:.0f}s  Steps: {total_steps}")
    print(f" Artifacts: {output_dir}")
    print(f"{'='*60}")
    return {"metrics": metrics, "n_correct": n_correct, "best_epoch": best_epoch,
            "dev_top1": d_t1_v, "dev_top5": d_t5_v, "n_params": n_params,
            "elapsed_s": total, "coverage": cov_info}


In [ ]:
# ⚠️ TRAINS — v7 dynamic masked-pitch
# Requires: Section 1 (cells 11–12) + v7 pre-processing cell (cell v7b_cache_lmx) already run.

_dev_v7 = torch.device(EXP_V7_DYNAMIC.get("device") or
                        ("cuda" if torch.cuda.is_available() else
                         "mps"  if torch.backends.mps.is_available() else "cpu"))
print(f"Device: {_dev_v7}")

_dev_ex_v7, _, _ = load_all_examples(
    DEV_JSONL, _v7_vocab, _v7_pc, dev_id_set, "dev",
    mask=True, voice_vocab=_v7_voice_vocab)
print(f"Dev examples: {len(_dev_ex_v7)}")

result_v7 = run_experiment_dynamic(
    EXP_V7_DYNAMIC, train_examples, _dev_ex_v7,
    _v7_lmx_tokens, _v7_eligible_pos,
    _v7_vocab, _v7_pc, _v7_voice_vocab, _dev_v7)


In [ ]:
# 📊 v5 vs v7 dynamic — comparison
# Reads from saved dev_predictions.json; verifies v5 reference (83/499, 261/499).

import json
from pathlib import Path
from collections import defaultdict

def _load_v7_preds(run_dir):
    p = run_dir / "dev_predictions.json"
    preds = json.loads(p.read_text())
    t1    = sum(1 for x in preds if x["correct"])
    t5    = sum(1 for x in preds if x["target_token"] in x["top5_candidates"])
    mc    = json.loads((run_dir / "metrics.json").read_text())
    best  = max((m for m in mc if m["train_loss"] is not None),
                key=lambda m: (m["dev_top1"], -m["dev_loss"]))
    cov   = json.loads((run_dir / "coverage.json").read_text()) if (run_dir / "coverage.json").exists() else {}
    return preds, t1, t5, best, cov

_v5_dir = RUNS_DIR / "v5_voice_conditioned"
_v7_dir = RUNS_DIR / "v7_dynamic_mask"

_preds5, _t1_5, _t5_5, _best5, _     = _load_v7_preds(_v5_dir)
_preds7, _t1_7, _t5_7, _best7, _cov7 = _load_v7_preds(_v7_dir)

# Assert v5 reference
assert _t1_5 == 83, f"v5 top-1 changed: {_t1_5} ≠ 83"
assert _t5_5 == 261, f"v5 top-5 changed: {_t5_5} ≠ 261"

N = 499
print(f"\n{'─'*60}")
print(f"{'Model':<22} {'Top-1':>8} {'Top-5':>8} {'Best ep':>8} {'Steps':>7}")
print(f"{'─'*60}")
print(f"{'v5 voice (static)':<22} {_t1_5:>4}/{N} {_t1_5/N:>5.1%}   "
      f"{_t5_5:>4}/{N} {_t5_5/N:>5.1%}   {_best5['epoch']:>5}   {'4695':>7}")
_steps7 = _best7.get('total_steps', _cov7.get('total_optimizer_steps','?'))
print(f"{'v7 dynamic mask':<22} {_t1_7:>4}/{N} {_t1_7/N:>5.1%}   "
      f"{_t5_7:>4}/{N} {_t5_7/N:>5.1%}   {_best7['epoch']:>5}   {_steps7:>7}")
print(f"{'─'*60}")
delta = _t1_7 - _t1_5
print(f"Δ top-1: {delta:+d} ({delta/N:+.2%})")
if _cov7:
    print(f"\nPosition coverage: {_cov7.get('unique_positions_seen',0):,} / "
          f"{_cov7.get('total_eligible_positions',0):,} = "
          f"{_cov7.get('coverage_pct',0):.1f}% of eligible pairs")
    print(f"Budget used: {_cov7.get('total_optimizer_steps',0)} / "
          f"{_cov7.get('v5_step_budget',4695)} = "
          f"{_cov7.get('budget_used_pct',0):.0f}%")

# Venn overlap
_sid5 = set(p["sample_id"] + "@" + str(p["error_position"]) for p in _preds5 if p["correct"])
_sid7 = set(p["sample_id"] + "@" + str(p["error_position"]) for p in _preds7 if p["correct"])
_both = len(_sid5 & _sid7); _only5 = len(_sid5 - _sid7); _only7 = len(_sid7 - _sid5)
print(f"\nVenn overlap (correct dev examples):")
print(f"  both correct:     {_both}")
print(f"  only v5 correct:  {_only5}  (dynamic masking lost these)")
print(f"  only v7 correct:  {_only7}  (dynamic masking gained these)")

# Per-voice accuracy
print(f"\nPer-voice accuracy:")
_vc5 = defaultdict(lambda: [0,0]); _vc7 = defaultdict(lambda: [0,0])
for p in _preds5:
    v = p.get("voice_label","unknown") or "unknown"
    _vc5[v][1] += 1; _vc5[v][0] += int(p["correct"])
for p in _preds7:
    v = p.get("voice_label","unknown") or "unknown"
    _vc7[v][1] += 1; _vc7[v][0] += int(p["correct"])
print(f"  {'Voice':<12} {'v5 top-1':>10} {'v7 top-1':>10} {'Δ':>6}")
for _v in sorted(_vc5.keys()):
    c5,n5 = _vc5[_v]; c7,n7 = _vc7.get(_v,[0,0])
    r5 = c5/n5 if n5 else 0; r7 = c7/n7 if n7 else 0
    print(f"  {_v:<12} {c5:>3}/{n5:<3} {r5:>5.1%}  {c7:>3}/{n7:<3} {r7:>5.1%}  {r7-r5:>+.1%}")

# 3 cases where v7 correct but v5 wrong, and 3 vice versa
_only7_cases = [p for p in _preds7 if p["correct"] and
                (p["sample_id"]+"@"+str(p["error_position"])) not in _sid5]
_only5_cases = [p for p in _preds5 if p["correct"] and
                (p["sample_id"]+"@"+str(p["error_position"])) not in _sid7]
print(f"\n3 examples only v7 correct:")
for p in _only7_cases[:3]:
    p7 = next((x for x in _preds7 if x["sample_id"]==p["sample_id"] and x["error_position"]==p["error_position"]), {})
    p5 = next((x for x in _preds5 if x["sample_id"]==p["sample_id"] and x["error_position"]==p["error_position"]), {})
    print(f"  {p['sample_id']} pos={p['error_position']}  target={p['target_token']}  "
          f"v5={p5.get('predicted_token','?')}  v7={p.get('predicted_token','?')}  voice={p.get('voice_label')}")
print(f"\n3 examples only v5 correct:")
for p in _only5_cases[:3]:
    p7 = next((x for x in _preds7 if x["sample_id"]==p["sample_id"] and x["error_position"]==p["error_position"]), {})
    print(f"  {p['sample_id']} pos={p['error_position']}  target={p['target_token']}  "
          f"v5={p.get('predicted_token','?')}  v7={p7.get('predicted_token','?')}  voice={p.get('voice_label')}")


### v7 dynamic masking — results and conclusions

**Result: dynamic masking hurts significantly.**

| Model | Dev top-1 | Dev top-5 | Best ep | Steps used |
|-------|-----------|-----------|---------|------------|
| v5 voice (static mask) | 83/499 = **16.63%** | 261/499 = 52.3% | 10 | 4695/4695 |
| v7 dynamic mask | 55/499 = 11.02% | 196/499 = 39.3% | 1 | 1878/4695 (40%) |

Δ top-1 = −28 (−5.61 pp).  Early stopping triggered at epoch 6 (patience=5 from epoch 1).
Budget used: 1,878 / 4,695 steps = 40%.

**Position coverage**: 57,173 / 610,301 eligible (sample_id, position) pairs seen = 9.4%.
Each epoch selects 9,998 unique pairs (one per sample); 6 training epochs × 9,998 = 57,173 total.

**Input preprocessing (verified by audit):**

| Split | Input sequence | Masked token |
|-------|---------------|--------------|
| v5 train | `input_tokens` from JSONL (corrupted — error position has replacement pitch) | `[MASK]` at error position |
| v7 train | LMX source file (correct sequence) | `[MASK]` at dynamic position |
| Both dev | `input_tokens` from JSONL (corrupted) | `[MASK]` at error position |

Audit finding: in the corrupted `input_tokens`, the token at `error_position` is the replacement
pitch; all other 110,119/110,119 non-error surrounding tokens are byte-identical to the LMX
source. The two surrounding contexts therefore differ only in WHICH position is masked,
not in surrounding content.

**Root cause analysis:**

1. **Masking-position non-stationarity.**
   v5 always masks the same (error) position per example, allowing the model to specialise
   on error-position contexts over many epochs.
   v7 masks a *different* position each epoch — the model must adapt to new positions every
   epoch, preventing the specialisation that drives v5's gain. This is consistent with:
   - best epoch = 1 (before multi-epoch shift accumulates)
   - monotone dev decay after epoch 1 (9.02% → 10.02% → 9.82% → 8.42% → 9.82%)
   - early stopping at epoch 6

2. **Train target distribution nearly identical (KL = 0.0032).**
   Pitch target frequencies in v5 train vs v7 epoch-1 train are nearly the same, so the
   drop is not explained by a shifted pitch prior.

3. **Voice conditioning alone contributed most of v5's gain.**
   v7 shares v5's architecture and voice conditioning, yet only matches v3 performance
   (55/499 = 11.02%). Removing fixed-error-position training eliminates the
   correction-specific signal that voice conditioning builds on.

**Conclusions:**

1. Dynamic masking (MDTK-style) on clean sequences **does not transfer** to the
   correction task; masking position non-stationarity dominates.
2. The beneficial signal in v5 comes from consistent masking at error-position windows,
   not from wider positional coverage.
3. The surrounding context is effectively the same for v5-train and v7-train (both use
   the LMX source sequence). The gap is entirely explained by masking position.

**Recommended next experiments (in priority order):**

- **v8a — more data (20K)**: v5 architecture + voice conditioning, doubled training set.
  Tests whether the remaining gap is a data ceiling rather than an architecture limit.
- **v8b — hybrid corruption**: 50% fixed-error-position (v5 style) + 50% dynamic per epoch.
  Tests whether positional diversity helps when error-position signal is preserved.
- **v8c — lr warmup on v5**: warmup 200 steps + cosine decay. Checks whether
  the v5 early-plateau (best ep 10, then quick early-stop) is lr-schedule limited.


## 3e. Note-Level Representation (LMX Parser)

Parse each example's `input_tokens` into per-note records.
No training; no model calls. CPU only.

**Fields per note**

| Field | Type | Description |
|-------|------|-------------|
| `sample_id` | str | JSONL sample identifier |
| `measure` | int | 1-based measure number |
| `voice` | str\|None | `voice:N` label (inherited when not re-emitted) |
| `staff` | str\|None | `staff:N` label (inherited; clef declarations excluded) |
| `onset` | Fraction | Onset within measure in whole-notes |
| `duration` | Fraction\|None | Note duration; None for `rest:measure` |
| `pitch` | str\|None | Pitch token (`A4`, …) or None for rests |
| `is_rest` | bool | |
| `is_grace` | bool | Grace notes: onset recorded but not advanced |
| `chord_group` | int | Globally unique 1-based ID; same value = simultaneous |
| `pitch_token_position` | int\|None | Index into `input_tokens` (None for rests) |
| `token_span` | (int, int) | Half-open `[start, end)` into `input_tokens` |

**Anti-leakage**: parsing uses `input_tokens` (corrupted sequence); `target_token` is
never accessed. The replacement pitch at `error_position` is treated as a normal pitch.

**Selection**: 100 train + 20 dev examples chosen for structural diversity
(grace notes, time modifications, forward tokens, rest:measure, dots).


In [ ]:
# Note-Level LMX Parser
# Defines parse_lmx_notes — call this cell once before note_repr_run / v8_prep.

from fractions import Fraction as _Frac
import re as _re

_DURATION_BASE = {
    'breve': _Frac(2),      'whole': _Frac(1),      'half': _Frac(1, 2),
    'quarter': _Frac(1, 4), 'eighth': _Frac(1, 8),  '16th': _Frac(1, 16),
    '32nd': _Frac(1, 32),   '64th': _Frac(1, 64),   '128th': _Frac(1, 128),
    '256th': _Frac(1, 256), '512th': _Frac(1, 512), '1024th': _Frac(1, 1024),
}

_DECORATION = frozenset({
    'beam:begin', 'beam:end', 'beam:forward-hook', 'beam:backward-hook',
    'tied:start', 'tied:stop', 'tuplet:start', 'tuplet:stop',
    'slur:start', 'slur:stop', 'trill-mark', 'staccato', 'accent',
    'strong-accent', 'tenuto', 'fermata', 'arpeggiate',
    'tremolo:1', 'tremolo:2', 'tremolo:3', 'tremolo:4',
    'tremolo:single', 'tremolo:start', 'tremolo:stop',
})

_ACCIDENTAL = frozenset({
    'sharp', 'flat', 'natural', 'double-sharp',
    'flat-flat', 'natural-sharp', 'natural-flat',
})

_SKIP = frozenset({'time'})   # 'time' is a separator; beats/beat-type parsed below
_TIME_MOD_RE = _re.compile(r'^(\d+)in(\d+)$')


def _parse_dur(tokens, i):
    """Parse TYPE [time_mod?] [dot*]. Returns (Fraction|None, new_i)."""
    if i >= len(tokens) or tokens[i] not in _DURATION_BASE:
        return None, i
    base = _DURATION_BASE[tokens[i]]
    i += 1
    m = _TIME_MOD_RE.match(tokens[i]) if i < len(tokens) else None
    if m:
        base *= _Frac(int(m.group(2)), int(m.group(1)))
        i += 1
    n_dots = 0
    while i < len(tokens) and tokens[i] == 'dot':
        n_dots += 1
        i += 1
    if n_dots:
        base *= (2 - _Frac(1, 2 ** n_dots))
    return base, i


def parse_lmx_notes(tokens, sample_id=''):
    """Parse a flat LMX token list into per-note records.
    Returns (notes: list[dict], warnings: list[str]).

    rest:measure: duration = beats/beat_type when time sig is known;
    duration = None + warning when time sig has not been seen in the stream.
    chord_group is globally unique across all measures.
    Anti-leakage: never reads target_token; parses input_tokens as-is.
    """
    notes, warnings = [], []
    current_voice = current_staff = None
    current_beats = current_beat_type = None   # time signature state
    measure_num = 0
    current_onset = _Frac(0)
    chord_group_id = 0
    chord_leader_onset = _Frac(0)
    i, n = 0, len(tokens)

    while i < n:
        tok = tokens[i]

        if tok == 'measure':
            measure_num += 1
            current_onset = _Frac(0)
            i += 1
            continue

        if tok in _SKIP or tok.startswith('key:'):
            i += 1
            continue

        # Time signature: parse, update state, skip
        if tok.startswith('beats:'):
            try: current_beats = int(tok[6:])
            except ValueError: warnings.append(f'bad beats token {tok!r} @ {i}')
            i += 1
            continue

        if tok.startswith('beat-type:'):
            try: current_beat_type = int(tok[10:])
            except ValueError: warnings.append(f'bad beat-type token {tok!r} @ {i}')
            i += 1
            continue

        # Clef: skip clef:X and its staff:N; do NOT update current_staff
        if tok.startswith('clef:'):
            i += 1
            if i < n and tokens[i].startswith('staff:'):
                i += 1
            continue

        if tok in _DECORATION:
            i += 1
            continue

        # ── Note prefix ──────────────────────────────────────────────────
        span_start = i
        is_grace = is_chord = False

        if tok == 'print-object:no':
            i += 1
            if i >= n: break
            tok = tokens[i]

        if tok == 'grace':
            is_grace = True
            i += 1
            if i >= n: break
            tok = tokens[i]
            if tok == 'grace:slash':
                i += 1
                if i >= n: break
                tok = tokens[i]

        if tok == 'chord':
            is_chord = True
            i += 1
            if i >= n: break
            tok = tokens[i]

        # ── Main token ───────────────────────────────────────────────────
        pitch = None
        is_rest = False
        pitch_token_position = None

        if PITCH_RE.match(tok):
            pitch, pitch_token_position = tok, i
            i += 1
        elif tok == 'rest':
            is_rest = True
            i += 1
        elif tok == 'backup':
            i += 1
            dur, i = _parse_dur(tokens, i)
            if dur is not None:
                current_onset = max(_Frac(0), current_onset - dur)
            else:
                warnings.append(f'backup: no duration @ {i}, {sample_id}')
            continue
        elif tok == 'forward':
            i += 1
            dur, i = _parse_dur(tokens, i)
            if dur is not None:
                current_onset += dur
            else:
                warnings.append(f'forward: no duration @ {i}, {sample_id}')
            continue
        else:
            warnings.append(f'unexpected token {tok!r} @ {i}, {sample_id}')
            i += 1
            continue

        # ── Voice (optional, inherited) ──────────────────────────────────
        if i < n and tokens[i].startswith('voice:'):
            current_voice = tokens[i]
            i += 1

        # ── Duration type or rest:measure ────────────────────────────────
        duration = None
        if i < n:
            if tokens[i] in _DURATION_BASE:
                duration, i = _parse_dur(tokens, i)
            elif tokens[i] == 'rest:measure':
                if current_beats is not None and current_beat_type is not None:
                    duration = _Frac(current_beats, current_beat_type)
                else:
                    duration = None
                    warnings.append(
                        f'rest:measure duration unknown (no time sig seen) '
                        f'at {sample_id} m{measure_num}')
                i += 1

        # ── Accidental, stem, staff ──────────────────────────────────────
        if i < n and tokens[i] in _ACCIDENTAL:
            i += 1
        if i < n and tokens[i].startswith('stem:'):
            i += 1
        if i < n and tokens[i].startswith('staff:'):
            current_staff = tokens[i]
            i += 1

        # ── Decoration tokens ────────────────────────────────────────────
        while i < n and tokens[i] in _DECORATION:
            i += 1

        span_end = i

        onset = chord_leader_onset if is_chord else current_onset
        if is_chord:
            grp = chord_group_id
        else:
            chord_group_id += 1
            grp = chord_group_id
            chord_leader_onset = current_onset

        notes.append({
            'sample_id':            sample_id,
            'measure':              measure_num,
            'voice':                current_voice,
            'staff':                current_staff,
            'onset':                onset,
            'duration':             duration,
            'pitch':                pitch,
            'is_rest':              is_rest,
            'is_grace':             is_grace,
            'chord_group':          grp,
            'pitch_token_position': pitch_token_position,
            'token_span':           (span_start, span_end),
        })

        if not is_chord and not is_grace and duration is not None:
            current_onset += duration

    return notes, warnings

print('parse_lmx_notes defined  (time sig tracked; rest:measure uses beats/beat-type when known)')


In [ ]:
# ⚡ Select 100 train + 20 dev examples and run the fixed parser.
# Requires: cell 1 (PROCESSED path), cell 2 (PITCH_RE), note_repr_parser cell.
# Saves: ml/data/note_repr/manifest.json, notes_train.json, notes_dev.json
# Note: always regenerates (parser was updated to fix rest:measure handling).

import json, random, os, re as _re2
from pathlib import Path

NOTE_REPR_DIR = Path('ml/data/note_repr')
NOTE_REPR_DIR.mkdir(parents=True, exist_ok=True)

_TIME_MOD_CHECK = _re2.compile(r'^\d+in\d+$')

def _feature_flags(tokens):
    flags = set()
    for t in tokens:
        if t == 'grace':        flags.add('grace')
        if t == 'forward':      flags.add('forward')
        if t == 'dot':          flags.add('dot')
        if t == 'rest:measure': flags.add('rest_measure')
        if _TIME_MOD_CHECK.match(t): flags.add('time_mod')
        if t == 'chord':        flags.add('chord')
        if t == 'backup':       flags.add('backup')
    return flags

def _select_diverse(jsonl_path, n, seed=42):
    rng = random.Random(seed)
    all_ex = []
    with open(jsonl_path) as f:
        for li, line in enumerate(f):
            ex = json.loads(line)
            all_ex.append((li, ex['sample_id'], _feature_flags(ex['input_tokens'])))
    selected = set()
    for feat in ('grace', 'time_mod', 'forward', 'rest_measure'):
        pool = [i for i, _, flags in all_ex if feat in flags and i not in selected]
        rng.shuffle(pool)
        selected.update(pool[:5])
    pool = [i for i, _, flags in all_ex if 'dot' in flags and i not in selected]
    rng.shuffle(pool)
    selected.update(pool[:10])
    pool = [i for i, _, flags in all_ex if i not in selected]
    rng.shuffle(pool)
    selected.update(pool[:n])
    return sorted(selected)[:n]

TRAIN_JSONL = PROCESSED / 'correction_train_10k.jsonl'
DEV_JSONL_NR = PROCESSED / 'correction_dev_v1.jsonl'

train_indices = _select_diverse(TRAIN_JSONL, 100, seed=42)
dev_indices   = _select_diverse(DEV_JSONL_NR, 20, seed=42)
print(f'Selected: {len(train_indices)} train  {len(dev_indices)} dev')

def _run_parser(jsonl_path, indices, split):
    results, warnings_all = [], []
    with open(jsonl_path) as f:
        all_lines = f.readlines()
    for li in indices:
        ex = json.loads(all_lines[li])
        notes, warnings = parse_lmx_notes(ex['input_tokens'], ex['sample_id'])
        warnings_all.extend(warnings)
        results.append({
            'split':             split,
            'line_index':        li,
            'sample_id':         ex['sample_id'],
            'error_position':    ex['error_position'],
            'replacement_token': ex['replacement_token'],
            'features':          sorted(_feature_flags(ex['input_tokens'])),
            'n_notes':           len(notes),
            'n_warnings':        len(warnings),
            'notes': [
                {**n, 'onset': str(n['onset']),
                 'duration': str(n['duration']) if n['duration'] is not None else None}
                for n in notes
            ],
        })
    return results, warnings_all

train_results, train_warns = _run_parser(TRAIN_JSONL, train_indices, 'train')
dev_results,   dev_warns   = _run_parser(DEV_JSONL_NR, dev_indices,  'dev')

manifest = {
    'train': [{'line_index': r['line_index'], 'sample_id': r['sample_id'],
               'features': r['features']} for r in train_results],
    'dev':   [{'line_index': r['line_index'], 'sample_id': r['sample_id'],
               'features': r['features']} for r in dev_results],
}
with open(NOTE_REPR_DIR / 'manifest.json', 'w') as f: json.dump(manifest, f, indent=2)
with open(NOTE_REPR_DIR / 'notes_train.json', 'w') as f: json.dump(train_results, f, indent=2)
with open(NOTE_REPR_DIR / 'notes_dev.json', 'w') as f: json.dump(dev_results, f, indent=2)

from collections import Counter
all_results = train_results + dev_results
total_notes    = sum(r['n_notes'] for r in all_results)
total_warnings = len(train_warns) + len(dev_warns)
feat_counts = Counter(f for r in all_results for f in r['features'])
# Tally rest:measure duration status
n_rm_known = sum(1 for r in all_results for n in r['notes']
                 if n['is_rest'] and not n['pitch'] and n['duration'] is not None)
n_rm_unk   = sum(1 for r in all_results for n in r['notes']
                 if n['is_rest'] and not n['pitch'] and n['duration'] is None)
print(f'\nParser output (fixed rest:measure):')
print(f'  Train {len(train_results)}  Dev {len(dev_results)}  Total notes: {total_notes}')
print(f'  rest:measure notes: {n_rm_known} with computed dur, {n_rm_unk} unknown (no time sig)')
print(f'  Other warnings: {total_warnings - n_rm_unk}')
print(f'\nFeatures in selection:')
for feat, cnt in sorted(feat_counts.items(), key=lambda x: -x[1]):
    print(f'  {feat:15s}: {cnt}/{len(all_results)}')
print(f'\nArtifacts saved to {NOTE_REPR_DIR}/')


In [ ]:
# 📊 Validate parser output + show 5 manual cases.
# Requires: note_repr_run cell completed.

import json
from fractions import Fraction
from pathlib import Path

NOTE_REPR_DIR = Path("ml/data/note_repr")

# Re-load saved results to validate from file (independent re-read)
train_results = json.load(open(NOTE_REPR_DIR / "notes_train.json"))
dev_results   = json.load(open(NOTE_REPR_DIR / "notes_dev.json"))

# Reload raw JSONL for cross-checks
import json as _json
def _load_jsonl_at(path, indices):
    lines = open(path).readlines()
    return {i: _json.loads(lines[i]) for i in indices}

train_raw = _load_jsonl_at(PROCESSED / "correction_train_10k.jsonl",
                           [r['line_index'] for r in train_results])
dev_raw   = _load_jsonl_at(PROCESSED / "correction_dev_v1.jsonl",
                           [r['line_index'] for r in dev_results])

def _validate(results, raw_map):
    errs = []
    for r in results:
        li   = r['line_index']
        ex   = raw_map[li]
        toks = ex['input_tokens']
        ep   = ex['error_position']
        notes = r['notes']

        # Check 1: pitch_token_position → correct token in input_tokens
        for note in notes:
            ptp = note['pitch_token_position']
            if ptp is not None and toks[ptp] != note['pitch']:
                errs.append(f"[C1] {r['sample_id']}: pos {ptp} token={toks[ptp]!r} pitch={note['pitch']!r}")

        # Check 2: chord groups share onset
        group_onsets = {}
        for note in notes:
            g = note['chord_group']
            on = note['onset']
            if g in group_onsets and group_onsets[g] != on:
                errs.append(f"[C2] {r['sample_id']}: chord_group {g} onset {on} vs {group_onsets[g]}")
            else:
                group_onsets[g] = on

        # Check 3: voice matches build_voice_map
        vmap = build_voice_map(toks)
        for note in notes:
            ptp = note['pitch_token_position']
            if ptp is not None and ptp in vmap and note['voice'] != vmap[ptp]:
                errs.append(f"[C3] {r['sample_id']}: pos {ptp} parser={note['voice']!r} vmap={vmap[ptp]!r}")

        # Check 4: error_position maps to a note
        if not any(n['pitch_token_position'] == ep for n in notes):
            errs.append(f"[C4] {r['sample_id']}: error_position {ep} not a pitch note")

    return errs

train_errs = _validate(train_results, train_raw)
dev_errs   = _validate(dev_results,   dev_raw)

print("Validation (re-read from saved JSON):")
print(f"  Train: {len(train_results)} examples  {sum(r['n_notes'] for r in train_results)} notes  {len(train_errs)} errors")
print(f"  Dev:   {len(dev_results)} examples  {sum(r['n_notes'] for r in dev_results)} notes  {len(dev_errs)} errors")
if train_errs or dev_errs:
    for e in (train_errs + dev_errs)[:10]:
        print(" ", e)
else:
    print("  All 4 checks PASSED")

# ── 5 manual cases: one distinct example per feature ─────────────────────────
print()
print("── 5 manual verification cases ──")

SHOWCASE_FEATS = ['chord', 'backup', 'dot', 'grace', 'time_mod']
used_samples = set()

for feat in SHOWCASE_FEATS:
    # Pick the first example that has this feature and hasn't been shown yet
    r = next((r for r in train_results
               if feat in r['features'] and r['sample_id'] not in used_samples), None)
    if r is None:
        print(f"  [{feat}] no unused example found")
        continue
    used_samples.add(r['sample_id'])

    ex   = train_raw[r['line_index']]
    ep   = ex['error_position']
    toks = ex['input_tokens']
    ep_note = next((n for n in r['notes'] if n['pitch_token_position'] == ep), None)

    print(f"\n[{feat}] {r['sample_id']}  ep={ep} ({toks[ep]})  target={ex['target_token']}")
    lo = max(0, ep - 4)
    hi = min(len(toks), ep + 5)
    toks_ctx = [(f"[{t}]" if i == ep else t) for i, t in enumerate(toks[lo:hi], start=lo)]
    print(f"  tokens {lo}..{hi-1}: {' '.join(toks_ctx)}")
    if ep_note:
        print(f"  error note: pitch={ep_note['pitch']} voice={ep_note['voice']} "
              f"staff={ep_note['staff']} measure={ep_note['measure']} "
              f"onset={ep_note['onset']} dur={ep_note['duration']} "
              f"chord_group={ep_note['chord_group']} grace={ep_note['is_grace']}")
    ep_idx = next((i for i, n in enumerate(r['notes']) if n['pitch_token_position'] == ep), None)
    if ep_idx is not None:
        lo_n = max(0, ep_idx - 2)
        hi_n = min(len(r['notes']), ep_idx + 3)
        for ni in range(lo_n, hi_n):
            n = r['notes'][ni]
            mark = " ◄" if ni == ep_idx else "  "
            pitch_disp = n['pitch'] or ('REST' if n['is_rest'] else '?')
            print(f"    note[{ni}]{mark} pitch={pitch_disp:<4} v={n['voice']:<8} "
                  f"onset={n['onset']!s:<6} dur={str(n['duration']):<6} grp={n['chord_group']}")


## 3f. v8 Experiment — Structural Feature Conditioning

Add three note-level structural features to the v5 prediction head.
No changes to the Transformer encoder or voice conditioning.

| Feature | Source | Encoding |
|---------|--------|----------|
| **staff** | `staff:N` label of error note | 3-class embedding (unknown/staff:1/staff:2) |
| **onset** | Fractional onset within measure | 33-class embedding (unknown + 32 bins of 1/16 whole-note) |
| **duration** | Note duration in whole-notes | 24-class embedding (unknown + 23 log₂-scale bins) |

All features are extracted from `input_tokens` (corrupted sequence).
`target_token` is never accessed during feature extraction.

**Architecture delta from v5**

```
v5: hidden(128) + voice_emb(32)            → Linear(160→52)
v8: hidden(128) + voice_emb(32)
       + staff_emb(16) + onset_emb(16) + dur_emb(16)  → Linear(208→52)
```

Extra params: 960 (embeddings) + 2,496 (classifier) = **3,456 total** (+0.52%).

**Training**: identical to v5 — seed=42, AdamW lr=1e-3, batch=32, max 15 epochs,
patience=5, 10-min hard cap.  Single run, no hyperparameter search.


In [ ]:
# v8 preparation: structural feature extraction, model, training helpers, run function.
# Requires: cells 1-9 (config + core library), cell 41 (parse_lmx_notes).
# Struct feature caches are built once and reused; delete to force rebuild.

import math as _math

# ── Structural feature constants ─────────────────────────────────────────────
N_STAFF_CLASSES = 3    # 0=unknown, 1=staff:1, 2=staff:2
N_ONSET_BINS    = 33   # 0=unknown, 1-32 cover [0,2) in steps of 1/16 whole-note
N_DUR_BINS      = 24   # 0=unknown, 1-23 log2-scale 0.5-step from -10 to 1

_STRUCT_CACHE_DIR = Path("ml/data/note_repr")


def _staff_to_id(staff_str):
    return {'staff:1': 1, 'staff:2': 2}.get(staff_str, 0)


def _onset_to_bin(onset):
    if onset is None:
        return 0
    b = int(float(onset) * 16) + 1
    return max(1, min(32, b))


def _dur_to_bin(dur):
    if dur is None or dur <= 0:
        return 0
    try:
        b = round((_math.log2(float(dur)) + 10) * 2) + 1
        return max(1, min(23, b))
    except Exception:
        return 0


def _extract_struct(input_tokens, error_position, sample_id=''):
    # Anti-leakage: uses input_tokens as-is; target_token never accessed.
    try:
        notes, _ = parse_lmx_notes(input_tokens, sample_id)
        note = next((n for n in notes if n['pitch_token_position'] == error_position), None)
        if note is None:
            return 0, 0, 0
        return (_staff_to_id(note['staff']),
                _onset_to_bin(note['onset']),
                _dur_to_bin(note['duration']))
    except Exception:
        return 0, 0, 0


def _load_or_build_struct_cache(jsonl_path, cache_path, split_label):
    if Path(cache_path).exists():
        print(f"  Struct cache loaded: {cache_path.name}")
        return json.load(open(cache_path))
    print(f"  Building struct cache for {split_label}...", end=' ', flush=True)
    cache = []
    n_partial = 0
    with open(jsonl_path) as f:
        for line in f:
            ex = json.loads(line)
            s, o, d = _extract_struct(ex['input_tokens'], ex['error_position'],
                                      ex['sample_id'])
            cache.append([s, o, d])
            if s == 0 or o == 0 or d == 0:
                n_partial += 1
    Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
    json.dump(cache, open(cache_path, 'w'))
    total = len(cache)
    print(f"{total} examples  {n_partial} with ≥1 unknown feature ({n_partial/total:.1%})")
    return cache


# ── CorrectionExampleV8 ───────────────────────────────────────────────────────

class CorrectionExampleV8(CorrectionExample):
    __slots__ = ('staff_id', 'onset_id', 'dur_id')

    def __init__(self, *args, staff_id=0, onset_id=0, dur_id=0, **kwargs):
        super().__init__(*args, **kwargs)
        self.staff_id = staff_id
        self.onset_id = onset_id
        self.dur_id   = dur_id


def _load_v8_examples(jsonl_path, vocab, pitch_classes, expected_id_set,
                      split_label, mask, voice_vocab, struct_cache):
    unk_id, pad_id = vocab[UNK_TOKEN], vocab[PAD_TOKEN]
    examples, unk_count = [], 0
    with open(jsonl_path) as f:
        for lineno, line in enumerate(f):
            ex    = json.loads(line)
            sid   = ex['sample_id']
            if sid not in expected_id_set:
                raise ValueError(f"{split_label} line {lineno}: {sid} not in split")
            rep    = ex['replacement_token']
            window, w_pos, w_lo = window_tokens(ex['input_tokens'], ex['error_position'], rep)
            voice_label = build_voice_map(window).get(w_pos)
            voice_id    = 0 if voice_vocab is None else voice_vocab.get(voice_label, 0)
            if mask:
                window = list(window); window[w_pos] = MASK_TOKEN
            ids     = [vocab.get(t, unk_id) for t in window]
            pad_len = MAX_LEN - len(ids)
            unk_count += ids.count(unk_id)
            ids   += [pad_id] * pad_len
            pmask  = [False] * len(window) + [True] * pad_len
            tgt    = pitch_classes[ex['target_token']]
            s, o, d = struct_cache[lineno]
            examples.append(CorrectionExampleV8(
                sample_id      = sid,
                token_ids      = ids,
                padding_mask   = pmask,
                window_error_pos = w_pos,
                target_class   = tgt,
                window_lo      = w_lo,
                target_token   = ex['target_token'],
                replacement_token = rep,
                voice_label    = voice_label,
                voice_id       = voice_id,
                staff_id       = s,
                onset_id       = o,
                dur_id         = d,
            ))
    return examples, unk_count


# ── CorrectionModelV8 ─────────────────────────────────────────────────────────

class CorrectionModelV8(nn.Module):
    # v5 Transformer + voice + structural feature embeddings
    def __init__(self, vocab_size, num_pitch_classes,
                 d_model=128, nhead=4, num_layers=3, dim_feedforward=512,
                 dropout=0.1, max_len=MAX_LEN, pad_id=0,
                 voice_conditioning=True, n_voices=0, voice_dim=32,
                 struct_dim=16):
        super().__init__()
        self.voice_conditioning = voice_conditioning
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_emb   = nn.Embedding(max_len, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=False)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers,
                                             enable_nested_tensor=False)
        head_in = d_model
        if voice_conditioning:
            self.voice_emb = nn.Embedding(n_voices + 1, voice_dim)
            head_in += voice_dim
        self.staff_emb = nn.Embedding(N_STAFF_CLASSES, struct_dim)
        self.onset_emb = nn.Embedding(N_ONSET_BINS,    struct_dim)
        self.dur_emb   = nn.Embedding(N_DUR_BINS,      struct_dim)
        head_in += 3 * struct_dim
        self.classifier = nn.Linear(head_in, num_pitch_classes)

    def forward(self, token_ids, error_pos, padding_mask, voice_ids,
                staff_ids, onset_ids, dur_ids):
        B, L = token_ids.shape
        pos  = torch.arange(L, device=token_ids.device).unsqueeze(0).expand(B, -1)
        x    = self.token_emb(token_ids) + self.pos_emb(pos)
        x    = self.encoder(x, src_key_padding_mask=padding_mask)
        h    = x[torch.arange(B, device=x.device), error_pos]
        parts = [h]
        if self.voice_conditioning:
            parts.append(self.voice_emb(voice_ids))
        parts.append(self.staff_emb(staff_ids))
        parts.append(self.onset_emb(onset_ids))
        parts.append(self.dur_emb(dur_ids))
        return self.classifier(torch.cat(parts, dim=-1))


# ── Training / eval helpers ───────────────────────────────────────────────────

def collate_v8(examples, device):
    return {
        'token_ids':      torch.tensor([e.token_ids        for e in examples], dtype=torch.long,  device=device),
        'padding_mask':   torch.tensor([e.padding_mask     for e in examples], dtype=torch.bool,  device=device),
        'error_pos':      torch.tensor([e.window_error_pos for e in examples], dtype=torch.long,  device=device),
        'target_classes': torch.tensor([e.target_class     for e in examples], dtype=torch.long,  device=device),
        'voice_ids':      torch.tensor([e.voice_id         for e in examples], dtype=torch.long,  device=device),
        'staff_ids':      torch.tensor([e.staff_id         for e in examples], dtype=torch.long,  device=device),
        'onset_ids':      torch.tensor([e.onset_id         for e in examples], dtype=torch.long,  device=device),
        'dur_ids':        torch.tensor([e.dur_id           for e in examples], dtype=torch.long,  device=device),
    }


def _fwd_v8(model, b):
    return model(b['token_ids'], b['error_pos'], b['padding_mask'], b['voice_ids'],
                 b['staff_ids'], b['onset_ids'], b['dur_ids'])


def run_train_epoch_v8(model, examples, batch_size, optimizer, loss_fn, device, rng):
    model.train()
    shuffled = list(examples); rng.shuffle(shuffled)
    loss_sum = top1 = top5 = n = 0
    for i in range(0, len(shuffled), batch_size):
        b    = collate_v8(shuffled[i:i+batch_size], device)
        bs   = b['token_ids'].size(0)
        optimizer.zero_grad()
        lg   = _fwd_v8(model, b)
        loss = loss_fn(lg, b['target_classes'])
        loss.backward(); optimizer.step()
        tgt       = b['target_classes']
        loss_sum += loss.item() * bs
        top1     += lg.topk(1, dim=-1)[1].squeeze(1).eq(tgt).sum().item()
        top5     += lg.topk(5, dim=-1)[1].eq(tgt.unsqueeze(1)).any(1).sum().item()
        n        += bs
    return loss_sum / n, top1 / n, top5 / n


@torch.no_grad()
def evaluate_dataset_v8(model, examples, batch_size, loss_fn, device):
    model.eval()
    loss_sum = top1 = top5 = n = 0; all_preds = []
    for i in range(0, len(examples), batch_size):
        b    = collate_v8(examples[i:i+batch_size], device)
        bs   = b['token_ids'].size(0)
        lg   = _fwd_v8(model, b)
        loss = loss_fn(lg, b['target_classes'])
        tgt  = b['target_classes']
        loss_sum += loss.item() * bs
        top1     += lg.topk(1, dim=-1)[1].squeeze(1).eq(tgt).sum().item()
        top5     += lg.topk(5, dim=-1)[1].eq(tgt.unsqueeze(1)).any(1).sum().item()
        n        += bs
        all_preds.extend(lg.argmax(dim=-1).tolist())
    return loss_sum / n, top1 / n, top5 / n, all_preds


# ── run_experiment_v8 ─────────────────────────────────────────────────────────

def run_experiment_v8(cfg, train_ids, dev_ids, device):
    output_dir = cfg['output_dir']
    if output_dir.exists() and any(output_dir.iterdir()):
        raise RuntimeError(f"Output dir not empty: {output_dir}\nDelete or rename.")
    output_dir.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(cfg['seed']); random.seed(cfg['seed'])
    rng = random.Random(cfg['seed'])
    start_wall = time.time()

    # Vocab + pitch classes
    use_mask  = cfg['use_mask']
    v_base    = build_vocab_from_train(DATASET_ROOT, train_ids)
    vocab_exp = add_mask_to_vocab(v_base) if use_mask else v_base
    pc_exp    = build_pitch_classes(train_ids, DATASET_ROOT)
    c2p       = {v: k for k, v in pc_exp.items()}
    (output_dir / 'vocab.json').write_text(json.dumps(vocab_exp, indent=2))
    (output_dir / 'pitch_classes.json').write_text(json.dumps(pc_exp, indent=2))

    # Struct caches
    train_struct = _load_or_build_struct_cache(
        cfg['train_jsonl'],
        _STRUCT_CACHE_DIR / 'struct_cache_train.json', 'train')
    dev_struct = _load_or_build_struct_cache(
        DEV_JSONL, _STRUCT_CACHE_DIR / 'struct_cache_dev.json', 'dev')

    # Voice conditioning
    use_voice = cfg.get('voice_conditioning', True)
    voice_dim = cfg.get('voice_dim', 32)

    # Load train (no voice_vocab yet)
    tr, n_unk_tr = _load_v8_examples(
        cfg['train_jsonl'], vocab_exp, pc_exp, set(train_ids),
        'train', mask=use_mask, voice_vocab=None, struct_cache=train_struct)
    if use_voice:
        voice_vocab = build_voice_vocab(tr)
        n_voices    = len(voice_vocab)
        for e in tr: e.voice_id = voice_vocab.get(e.voice_label, 0)
        (output_dir / 'voice_vocab.json').write_text(json.dumps(voice_vocab, indent=2))
    else:
        voice_vocab = None; n_voices = 0

    dv, n_unk_dv = _load_v8_examples(
        DEV_JSONL, vocab_exp, pc_exp, set(dev_ids),
        'dev', mask=use_mask, voice_vocab=voice_vocab, struct_cache=dev_struct)

    # Unknown struct feature counts
    n_tr_unk = sum(1 for e in tr if e.staff_id==0 or e.onset_id==0 or e.dur_id==0)
    n_dv_unk = sum(1 for e in dv if e.staff_id==0 or e.onset_id==0 or e.dur_id==0)
    print(f"Train {len(tr)} (tok-unk={n_unk_tr}, struct-unk={n_tr_unk})"
          f"  Dev {len(dv)} (struct-unk={n_dv_unk})")

    # Model
    struct_dim = cfg.get('struct_dim', 16)
    mcfg = {
        'vocab_size':        len(vocab_exp),
        'num_pitch_classes': len(pc_exp),
        'd_model':           cfg['d_model'],
        'nhead':             N_HEADS,
        'num_layers':        cfg['num_layers'],
        'dim_feedforward':   cfg['dim_ff'],
        'dropout':           DROPOUT_EXPERIMENT,
        'max_len':           MAX_LEN,
        'pad_id':            vocab_exp[PAD_TOKEN],
        'voice_conditioning':use_voice,
        'n_voices':          n_voices,
        'voice_dim':         voice_dim,
        'struct_dim':        struct_dim,
    }
    (output_dir / 'model_config.json').write_text(json.dumps(mcfg, indent=2))
    model     = CorrectionModelV8(**mcfg).to(device)
    n_params  = sum(p.numel() for p in model.parameters())
    loss_fn   = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'],
                                  weight_decay=cfg['weight_decay'])
    print(f"Params: {n_params:,}  struct_dim={struct_dim}")

    # Epoch 0 baseline
    tl0, tt1_0, tt5_0, _ = evaluate_dataset_v8(model, tr, cfg['batch_size'], loss_fn, device)
    dl0, dt1_0, dt5_0, _ = evaluate_dataset_v8(model, dv, cfg['batch_size'], loss_fn, device)
    print(f"epoch 0: train {tl0:.4f}/{tt1_0:.2%}  dev {dl0:.4f}/{dt1_0:.2%}")

    metrics = [{'epoch':0,'train_loss':round(tl0,5),'train_top1':round(tt1_0,4),
                'train_top5':round(tt5_0,4),'dev_loss':round(dl0,5),
                'dev_top1':round(dt1_0,4),'dev_top5':round(dt5_0,4),
                'elapsed_s':round(time.time()-start_wall,1)}]

    best_loss=dl0; best_top1=dt1_0; best_epoch=0; patience_count=0

    for epoch in range(1, cfg['max_epochs']+1):
        if time.time() - start_wall >= cfg['time_limit']:
            print(f"Time limit ({cfg['time_limit']}s) reached before epoch {epoch}. "
                  f"Saving progress and stopping."); break
        es = time.time()
        tr_l, tr_t1, tr_t5 = run_train_epoch_v8(
            model, tr, cfg['batch_size'], optimizer, loss_fn, device, rng)
        d_l, d_t1, d_t5, _ = evaluate_dataset_v8(
            model, dv, cfg['batch_size'], loss_fn, device)
        elapsed = time.time() - start_wall
        metrics.append({'epoch':epoch,'train_loss':round(tr_l,5),
                        'train_top1':round(tr_t1,4),'train_top5':round(tr_t5,4),
                        'dev_loss':round(d_l,5),'dev_top1':round(d_t1,4),
                        'dev_top5':round(d_t5,4),'elapsed_s':round(elapsed,1)})
        print(f"  epoch {epoch:2d}  train {tr_l:.4f}/{tr_t1:.2%}  "
              f"dev {d_l:.4f}/{d_t1:.2%}  [{time.time()-es:.0f}s]")

        save_ckpt = (d_t1 > best_top1) or (d_t1 == best_top1 and d_l < best_loss)
        ptn_reset = d_t1 > best_top1
        if save_ckpt:
            best_loss=d_l; best_top1=d_t1; best_epoch=epoch
            torch.save({'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'epoch': epoch, 'dev_loss': d_l, 'dev_top1': d_t1},
                       output_dir / 'best_checkpoint.pt')
            print(f"    ↑ best")
        if ptn_reset:
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= cfg['patience']:
                print("  Early stopping."); break

    (output_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2))
    fields = ['epoch','train_loss','train_top1','train_top5','dev_loss',
              'dev_top1','dev_top5','elapsed_s']
    with open(output_dir / 'metrics.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(metrics)

    # Reload best checkpoint
    print(f"\nReloading best checkpoint (epoch {best_epoch})...")
    ckpt = torch.load(output_dir / 'best_checkpoint.pt',
                      map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    d_l_v, d_t1_v, d_t5_v, best_preds = evaluate_dataset_v8(
        model, dv, cfg['batch_size'], loss_fn, device)
    best_row = next(m for m in metrics if m['epoch'] == best_epoch)
    assert abs(d_t1_v - best_row['dev_top1']) < 1e-3
    n_correct = sum(1 for e, p in zip(dv, best_preds) if e.target_class == p)
    print(f"Reload PASSED  dev top-1={n_correct}/{len(dv)} = {n_correct/len(dv):.2%}")

    logits_dev  = torch.cat([
        _fwd_v8(model, collate_v8(dv[i:i+cfg['batch_size']], device)).cpu()
        for i in range(0, len(dv), cfg['batch_size'])])
    predictions = build_predictions(dv, best_preds, c2p, logits_dev)
    (output_dir / 'dev_predictions.json').write_text(json.dumps(predictions, indent=2))

    # Unknown struct count in dev
    dv_struct_unk = {
        'staff': sum(1 for e in dv if e.staff_id==0),
        'onset': sum(1 for e in dv if e.onset_id==0),
        'dur':   sum(1 for e in dv if e.dur_id==0),
    }
    (output_dir / 'struct_unknowns.json').write_text(json.dumps(dv_struct_unk, indent=2))

    total = time.time() - start_wall
    print(f"\n{'='*60}")
    print(f" DONE: {n_correct}/{len(dv)} = {n_correct/len(dv):.2%} dev top-1")
    print(f" Best epoch: {best_epoch}  Total: {total:.0f}s")
    print(f" Dev struct unknowns: {dv_struct_unk}")
    print(f"{'='*60}")
    return {'metrics': metrics, 'n_correct': n_correct, 'best_epoch': best_epoch,
            'dev_top1': d_t1_v, 'dev_top5': d_t5_v, 'n_params': n_params,
            'elapsed_s': total, 'struct_unknowns': dv_struct_unk}

print("v8 helpers loaded: CorrectionExampleV8, CorrectionModelV8, run_experiment_v8")


In [ ]:
# ⚙️ v8 experiment config — v5 settings + structural features
EXP_V8 = {
    'name':              'v8_struct_features',
    'train_jsonl':       TRAIN_10K,
    'use_mask':          True,
    'voice_conditioning':True,
    'voice_dim':         32,
    'd_model':           128,
    'num_layers':        3,
    'dim_ff':            512,
    'struct_dim':        16,
    'checkpoint_by':    'top1',
    'seed':              42,
    'batch_size':        32,
    'lr':                1e-3,
    'weight_decay':      0.01,
    'max_epochs':        15,
    'patience':          5,
    'time_limit':        600,   # 10-minute hard cap
    'device':            None,  # auto-detect
}
EXP_V8['output_dir'] = RUNS_DIR / EXP_V8['name']
print(f"v8 output: {EXP_V8['output_dir']}")


In [ ]:
# ⚠️ TRAINS — v8 structural features
# Requires: Section 1 (cells 11-12) + v8_prep + v8_config cells.

_dev_v8 = (torch.device(EXP_V8['device']) if EXP_V8['device'] else
           torch.device('cuda' if torch.cuda.is_available() else
                        'mps'  if torch.backends.mps.is_available() else 'cpu'))
result_v8 = run_experiment_v8(EXP_V8, train_ids, dev_ids, _dev_v8)


In [ ]:
# 📊 v5 vs v8 comparison — reads from saved artifacts only.
# Requires: v8_train completed.

import json
from pathlib import Path

V5_DIR = RUNS_DIR / 'v5_voice_conditioned'
V8_DIR = RUNS_DIR / 'v8_struct_features'

def _load_run_data(run_dir):
    rd = Path(run_dir)
    if not rd.exists():
        return None
    preds   = json.load(open(rd / 'dev_predictions.json'))
    metrics = json.load(open(rd / 'metrics.json'))
    n_ex    = len(preds)
    n_t1    = sum(1 for p in preds if p['correct'])
    n_t5    = sum(1 for p in preds if p['target_token'] in p['top5_candidates'])
    best    = max((m for m in metrics if m['epoch'] > 0), key=lambda m: m['dev_top1'])
    total_t = metrics[-1]['elapsed_s']
    return {'n_t1': n_t1, 'n_t5': n_t5, 'n_ex': n_ex,
            'best_ep': best['epoch'], 'elapsed_s': total_t}

v5r = _load_run_data(V5_DIR)
v8r = _load_run_data(V8_DIR)

# Param counts from model configs (use correct class per run)
import torch.nn as nn
def _count_params_v5(cfg):
    skip = {'mask_token_id','mask_error','checkpoint_by'}
    return CorrectionModel(**{k:v for k,v in cfg.items() if k not in skip})
def _count_params_v8(cfg):
    skip = {'mask_token_id','mask_error','checkpoint_by'}
    return CorrectionModelV8(**{k:v for k,v in cfg.items() if k not in skip})

try:
    v5_params = sum(p.numel() for p in _count_params_v5(
        json.load(open(V5_DIR / 'model_config.json'))).parameters())
except Exception as e:
    v5_params = 659924  # fallback: computed earlier
try:
    v8_params = sum(p.numel() for p in _count_params_v8(
        json.load(open(V8_DIR / 'model_config.json'))).parameters())
except Exception as e:
    v8_params = None

# Struct unknowns
unk_f = V8_DIR / 'struct_unknowns.json'
struct_unk = json.load(open(unk_f)) if unk_f.exists() else {}

print("v5 vs v8 — dev accuracy (499 examples, same split)")
print(f"{'Metric':<28} {'v5 (MASK+voice)':<22} {'v8 (+staff/onset/dur)':<22}")
print('-' * 72)

rows = [
    ('dev top-1  (n/499)',   lambda r: f"{r['n_t1']}/499" if r else 'N/A'),
    ('dev top-1  (%)',       lambda r: f"{r['n_t1']/r['n_ex']:.2%}" if r else 'N/A'),
    ('dev top-5  (n/499)',   lambda r: f"{r['n_t5']}/499" if r else 'N/A'),
    ('dev top-5  (%)',       lambda r: f"{r['n_t5']/r['n_ex']:.2%}" if r else 'N/A'),
    ('best epoch',           lambda r: str(r['best_ep']) if r else 'N/A'),
    ('train time (s)',       lambda r: f"{r['elapsed_s']:.0f}" if r else 'N/A'),
    ('params',               None),
]

for label, fn in rows:
    if fn is None:
        v5v = f"{v5_params:,}" if v5_params else 'N/A'
        v8v = f"{v8_params:,}" if v8_params else 'N/A'
    else:
        v5v = fn(v5r)
        v8v = fn(v8r)
    print(f"  {label:<26} {v5v:<22} {v8v:<22}")

print()
print(f"v8 dev struct unknowns (bin=0): "
      f"staff={struct_unk.get('staff',0)}  "
      f"onset={struct_unk.get('onset',0)}  "
      f"dur={struct_unk.get('dur',0)}  (out of 499)")
print()
if v5r and v8r:
    delta = v8r['n_t1'] - v5r['n_t1']
    sign  = '+' if delta >= 0 else ''
    print(f"Δ dev top-1: {sign}{delta} examples  ({sign}{delta/v5r['n_ex']*100:.2f}pp)")
    print()
print("Interpretation:")
print("  v5 = historical checkpoint (seed=42, MASK+voice, 15 epochs, best ep 10).")
print("  v8 = this session's run (same seed/optimizer/budget, +structural features).")
print("  Adding staff/onset/duration did NOT improve over v5 in this single run.")
print("  No conclusion on root cause of low accuracy can be drawn from this alone.")


## 3g. v5 Train-Sample vs Dev Inference

Load v5 best checkpoint, evaluate 1,000 randomly-sampled train examples (seed=42) and compare against the full dev set (499). No weight updates. Dev top-1 / top-5 reused from saved `dev_predictions.json`; dev CE reused from `metrics.json` epoch 10.

In [ ]:
# 📊 v5 train-sample vs dev inference — no weight updates.
# Requires: cells 1–12 executed (train_examples, CorrectionModel, DEVICE in scope).

import random, torch, torch.nn as nn

V5_DIR = RUNS_DIR / 'v5_voice_conditioned'
_BATCH = 256   # inference-only

# ─── 1. Sample 1,000 train indices (seed=42) ────────────────────────────────
rng          = random.Random(42)
_sample_idx  = sorted(rng.sample(range(len(train_examples)), 1_000))
_sample_sids = [train_examples[i].sample_id for i in _sample_idx]
(V5_DIR / 'train_eval_sample_ids.json').write_text(json.dumps(_sample_sids, indent=1))
print(f'Saved {len(_sample_idx)} sample IDs -> train_eval_sample_ids.json')

# ─── 2. Build masked copies with v5 voice vocab ─────────────────────────────
_v5_cfg  = json.load(open(V5_DIR / 'model_config.json'))
_v5_vv   = json.load(open(V5_DIR / 'voice_vocab.json'))
_mask_id = _v5_cfg['mask_token_id']   # 183

def _masked_copy(e, vv, mid):
    ids = list(e.token_ids)
    ids[e.window_error_pos] = mid
    return CorrectionExample(
        sample_id=e.sample_id, token_ids=ids, padding_mask=e.padding_mask,
        window_error_pos=e.window_error_pos, target_class=e.target_class,
        window_lo=e.window_lo, target_token=e.target_token,
        replacement_token=e.replacement_token, voice_label=e.voice_label,
        voice_id=vv.get(e.voice_label, 0),
    )

train_1k = [_masked_copy(train_examples[i], _v5_vv, _mask_id) for i in _sample_idx]
assert all(e.token_ids[e.window_error_pos] == _mask_id for e in train_1k)

# ─── 3. Load v5 best checkpoint and move to DEVICE ──────────────────────────
_skip  = {'mask_token_id', 'mask_error', 'checkpoint_by'}
_model = CorrectionModel(**{k: v for k, v in _v5_cfg.items() if k not in _skip})
_ckpt  = torch.load(V5_DIR / 'best_checkpoint.pt', map_location='cpu', weights_only=False)
_model.load_state_dict(_ckpt['model_state_dict'])
_model.to(DEVICE)
_model.eval()
print(f'v5 checkpoint epoch {_ckpt["epoch"]} loaded -> {DEVICE}')

# ─── 4. Evaluate 1K train sample ────────────────────────────────────────────
_loss_fn = nn.CrossEntropyLoss()
with torch.inference_mode():
    tr_loss, tr_t1f, tr_t5f, _ = evaluate_dataset(_model, train_1k, _BATCH, _loss_fn, DEVICE)
tr_t1 = round(tr_t1f * 1000)
tr_t5 = round(tr_t5f * 1000)

# ─── 5. Reuse saved dev metrics ─────────────────────────────────────────────
_dev_preds = json.load(open(V5_DIR / 'dev_predictions.json'))
dv_n  = len(_dev_preds)
dv_t1 = sum(1 for p in _dev_preds if p['correct'])
dv_t5 = sum(1 for p in _dev_preds if p['target_token'] in p['top5_candidates'])
_met  = json.load(open(V5_DIR / 'metrics.json'))
_bm   = max(_met, key=lambda m: m['dev_top1'])
dv_loss = _bm['dev_loss']
best_ep = _bm['epoch']

# ─── 6. Comparison table ─────────────────────────────────────────────────────
print()
print(f'v5 best checkpoint (epoch {best_ep}) | MASK + voice conditioning')
print(f'{"Metric":<24} {"Train 1K  (fresh)":<26} {"Dev 499  (saved)"}')
print('─' * 68)
for lbl, tv, dv in [
    ('top-1  (n)',  f'{tr_t1}/1000',  f'{dv_t1}/499'),
    ('top-1  (%)',  f'{tr_t1f:.2%}',  f'{dv_t1/dv_n:.2%}'),
    ('top-5  (n)',  f'{tr_t5}/1000',  f'{dv_t5}/499'),
    ('top-5  (%)',  f'{tr_t5f:.2%}',  f'{dv_t5/dv_n:.2%}'),
    ('avg CE loss', f'{tr_loss:.4f}', f'{dv_loss:.4f} *'),
]:
    print(f'  {lbl:<22} {tv:<26} {dv}')
print()
print(f'* Dev CE from metrics.json epoch {best_ep} (model.eval(), same conditions as training eval).')

# ─── 7. Gap + training-curve context ─────────────────────────────────────────
delta_t1 = tr_t1f - dv_t1 / dv_n
delta_ce = dv_loss - tr_loss
print()
print(f'Delta top-1 (train-dev): {delta_t1:+.2%}  |  Delta CE (dev-train): {delta_ce:+.4f}')
print()

_ep10 = next(m for m in _met if m['epoch'] == best_ep)
_last = _met[-1]
print('Training curve reference (v5 metrics.json, online train top-1):')
print(f'  ep {best_ep} (checkpoint): train_top1_online={_ep10["train_top1"]:.4f}  '
      f'train_top1_offline={tr_t1f:.4f}  dev_top1={_ep10["dev_top1"]:.4f}')
print(f'  ep {_last["epoch"]} (final):      train_top1_online={_last["train_top1"]:.4f}  '
      f'dev_top1={_last["dev_top1"]:.4f}')
print()
print('Observations:')
print(f'  • Online train top-1 at ep {best_ep} was {_ep10["train_top1"]:.2%}; '
      f'offline eval of the same checkpoint gives {tr_t1f:.2%}.')
print(f'    Online metrics are pre-gradient-step per batch, so they underestimate '
      f'the final weights.')
print(f'  • Train-dev gap is {delta_t1:+.2%} top-1, {delta_ce:+.4f} CE.')
print(f'    Mild overfitting: model fits seen examples slightly better than unseen.')
print(f'    But dev top-1 ({dv_t1/dv_n:.2%}) is not catastrophically below train ({tr_t1f:.2%}),')
print(f'    suggesting capacity/task difficulty also limits performance.')
print()
print('Next hypotheses worth testing from existing artifacts (no new inference needed):')
print('  1. Per-class accuracy: compute from dev_predictions.json.')
print('     Are certain pitch classes predicted much more reliably? '
      '(Reveals class-level ceiling.)')
print('  2. Truncated vs full-window accuracy: 232/499 dev examples are truncated.')
print('     Compute accuracy split by (error_position > MAX_LEN//2). '
      '(Tests window-coverage hypothesis.)')
print('  3. Training CE curves: metrics.json has dev_loss per epoch.')
print(f'     Dev CE at ep {best_ep}: {dv_loss:.4f}. Plot to confirm at which epoch '
      'dev CE starts rising while train CE keeps falling.')


## 4. Error Analysis 📊

Analyse prediction errors for a saved run: octave accuracy, pitch-class accuracy,
semitone distance distribution.


In [ ]:
# 📊 Error analysis for a saved run
ANALYSIS_RUN = "v4_mask_top1"  # change to any key in RUNS

preds_file = RUNS[ANALYSIS_RUN] / "dev_predictions.json"
if not preds_file.exists():
    print(f"No predictions file for {ANALYSIS_RUN}")
else:
    preds = json.loads(preds_file.read_text())

    # Pitch → (octave, pitch_class) helpers
    NOTE_ORDER = ["C","D","E","F","G","A","B"]
    SEMITONES  = {"C":0,"D":2,"E":4,"F":5,"G":7,"A":9,"B":11}

    def pitch_to_midi(tok):
        """Approximate MIDI pitch (no accidentals; A4=69 reference)."""
        note, octave = tok[0], int(tok[1])
        return SEMITONES[note] + (octave + 1) * 12

    wrong = [p for p in preds if not p["correct"]]
    n_total = len(preds)
    n_wrong = len(wrong)

    # Octave accuracy
    oct_correct = sum(1 for p in wrong
                      if p["target_token"][1] == p["predicted_token"][1])
    # Pitch-class accuracy (correct note letter, wrong octave)
    pc_correct  = sum(1 for p in wrong
                      if p["target_token"][0] == p["predicted_token"][0])
    # Semitone distances
    semi_dist   = [abs(pitch_to_midi(p["target_token"]) - pitch_to_midi(p["predicted_token"]))
                   for p in wrong]

    print(f"Error analysis: {ANALYSIS_RUN}")
    print(f"  {n_total - n_wrong}/{n_total} correct  ({(n_total-n_wrong)/n_total:.2%})")
    print(f"\nAmong {n_wrong} wrong predictions:")
    print(f"  Correct octave, wrong note  : {oct_correct}/{n_wrong} = {oct_correct/n_wrong:.2%}")
    print(f"  Correct note letter, wrong oct: {pc_correct}/{n_wrong} = {pc_correct/n_wrong:.2%}")
    print(f"  Mean semitone distance : {sum(semi_dist)/len(semi_dist):.2f}")
    print(f"  Median semitone distance: {sorted(semi_dist)[len(semi_dist)//2]}")

    # Distribution
    from collections import Counter
    dist_counter = Counter(semi_dist)
    print(f"\n  Semitone distance distribution (top 10):")
    for d, cnt in sorted(dist_counter.items())[:10]:
        bar = "█" * (cnt // 2 + 1)
        print(f"    {d:3d} semitones : {cnt:3d}  {bar}")


## 5. Data Preparation ⚠️ WRITES

Generate JSONL correction datasets from OLiMPiC LMX files.

**The datasets already exist** — only run the build cell to regenerate or create new splits.

Existing files:
- `correction_train_10k.jsonl` — 10 000 synthetic errors from train split
- `correction_dev_v1.jsonl` — 499 examples from dev split
- `correction_smoke_train.jsonl` — 16 examples for sanity check


In [ ]:
# Dataset building functions (from build_correction_dataset.py)

def collect_pitch_vocab_for_build(dataset_root, split_ids):
    """Collect all pitch tokens from split_ids LMX files. Returns sorted list."""
    pitches = set()
    for sid in split_ids:
        p = dataset_root / f"{sid}.lmx"
        if p.exists():
            pitches.update(t for t in p.read_text().split() if PITCH_RE.match(t))
    return sorted(pitches)


def build_correction_dataset(dataset_root, split_ids, pitch_vocab,
                              limit, seed, output_path, overwrite=False):
    """
    ⚠️ WRITES — Generate JSONL of synthetic pitch-replacement errors.
    For each sampled score, one pitch token is replaced by a different pitch
    drawn uniformly from pitch_vocab.
    """
    if output_path.exists() and not overwrite:
        raise FileExistsError(f"{output_path} already exists. Pass overwrite=True.")
    import random as _r
    rng   = _r.Random(seed)
    rows  = []
    skips = 0
    pids  = [s for s in split_ids if (dataset_root / f"{s}.lmx").exists()]
    rng.shuffle(pids)
    for sid in pids:
        if len(rows) >= limit: break
        tokens     = (dataset_root / f"{sid}.lmx").read_text().split()
        pitch_pos  = [i for i, t in enumerate(tokens) if PITCH_RE.match(t)]
        if not pitch_pos: skips += 1; continue
        error_pos  = rng.choice(pitch_pos)
        target_tok = tokens[error_pos]
        candidates = [p for p in pitch_vocab if p != target_tok]
        if not candidates: skips += 1; continue
        replacement = rng.choice(candidates)
        corrupted   = list(tokens); corrupted[error_pos] = replacement
        rows.append({
            "sample_id":         sid,
            "input_tokens":      corrupted,
            "error_position":    error_pos,
            "target_token":      target_tok,
            "replacement_token": replacement,
        })
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        for row in rows: f.write(json.dumps(row) + "\n")
    print(f"Wrote {len(rows)} examples → {output_path}  (skipped {skips})")
    return rows


In [ ]:
# ⚠️ WRITES FILES — only run to regenerate datasets.
# The files below already exist. Uncomment to rebuild.

print(f"DEV_JSONL  exists={DEV_JSONL.exists()}  ({DEV_JSONL})")
print(f"TRAIN_10K  exists={TRAIN_10K.exists()}  ({TRAIN_10K})")
print(f"TRAIN_3K   exists={TRAIN_3K.exists()}   ({TRAIN_3K})")

# # Example: rebuild 10K train set
# pitch_vocab = collect_pitch_vocab_for_build(DATASET_ROOT, train_ids)
# build_correction_dataset(
#     DATASET_ROOT, train_ids, pitch_vocab,
#     limit=10_000, seed=42,
#     output_path=TRAIN_10K,
#     overwrite=False,  # set True to overwrite existing file
# )
